# 02 — Capa Silver: limpieza, calidad y MERGE incremental
Solo se procesan los sismos indicados en los parámetros.
- **Acelerogramas:** se leen las cabeceras, se tipan las columnas, se filtra la señal y se calculan PGA, PGV y espectros. Los registros que no se pueden procesar van a la **cuarentena** con su motivo. Los que se procesan con avisos se conservan con la columna `problemas`.
- **Espectros y eventos:** se derivan de los registros válidos.
- Las tablas de sismos se actualizan con `MERGE INTO` (si el registro existe se actualiza, si no se inserta).
- **Maestros** (territorio, edificios, STEAD, Tainan, Manta, suelo no lineal y fragilidad): se procesan la primera vez o con `recargar_maestros = true`. Las filas descartadas van a `silver_calidad_log` y a cuarentena.

## 1. Setup — imports

In [ ]:
%pip install -q "numpy<2" geopandas rasterio h5py pypdf pyproj shapely seaborn

In [ ]:
# Reinicia Python para que el driver y los workers usen las mismas librerías.
dbutils.library.restartPython()

In [ ]:
# Librerías que usa el notebook.
import io
import re
import zlib
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import h5py
import numpy as np
import pandas as pd
import rasterio
import shapely
from pyproj import Transformer
from rasterio.features import rasterize
from scipy import ndimage
from scipy.integrate import cumulative_trapezoid
from scipy.optimize import minimize
from scipy.signal import butter, cont2discrete, detrend, lfilter, sosfiltfilt, ss2tf
from scipy.signal.windows import tukey
from scipy.stats import kurtosis, norm
from sklearn.ensemble import HistGradientBoostingRegressor

from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window

## 2. Catálogo, esquema y parámetros

In [ ]:
# Catálogo y esquema; las fechas y la recarga de maestros llegan como parámetros del job.
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

dbutils.widgets.text("fecha_desde", "2007-01-01")
dbutils.widgets.text("fecha_hasta", "2025-12-31")
dbutils.widgets.text("recargar_maestros", "false")
fecha_desde, fecha_hasta = dbutils.widgets.get("fecha_desde"), dbutils.widgets.get("fecha_hasta")
recargar_maestros = dbutils.widgets.get("recargar_maestros").lower() == "true"

In [ ]:
# Área de estudio y constantes del procesamiento de señales.
PROVINCIAS_ESTUDIO = ["LIMA", "CALLAO"]
DISTRITOS_ESTUDIO = {"150104": "BARRANCO", "150122": "MIRAFLORES", "150130": "SAN BORJA", "150131": "SAN ISIDRO",
                     "150140": "SANTIAGO DE SURCO", "150141": "SURQUILLO"}
ZONA_E030_LIMA = 4
EPSG_UTM_LIMA = 32718
SEED = 700
FILTRO_HZ = (0.1, 25.0)
AMORTIGUAMIENTO = 0.05
PERIODOS = np.round(np.geomspace(0.05, 4.0, 50), 4)
G_CM_S2 = 980.665

## 3. Funciones

In [ ]:
# Pasa una tabla pandas pequeña a Spark, con los nulos bien tratados.
def a_spark(pdf, esquema=None):
    pdf = pdf.copy()
    for c in pdf.columns:
        if isinstance(pdf[c].dtype, (pd.StringDtype, pd.ArrowDtype)):
            pdf[c] = pdf[c].astype(object)
        if pdf[c].dtype == object:
            pdf[c] = pdf[c].map(lambda v: v if v is None or isinstance(v, (str, bytes, list, dict, np.ndarray))
                                else (None if pd.isna(v) else str(v)))
    return spark.createDataFrame(pdf, esquema) if esquema is not None else spark.createDataFrame(pdf)


# Guarda un DataFrame como tabla Delta y cuenta sus filas.
def escribir(df, tabla, particion=None, modo="overwrite", contar=True):
    escritor = df.write.format("delta").mode(modo)
    if modo == "overwrite":
        escritor = escritor.option("overwriteSchema", "true")
    if particion:
        escritor = escritor.partitionBy(particion)
    escritor.saveAsTable(tabla)
    if contar:
        n = spark.table(tabla).count()
        print(f"  {tabla}: {n:,} filas")
        return n


# Filas y tamaño de las tablas con un prefijo.
def inventario(prefijo):
    filas = []
    for t in spark.sql("SHOW TABLES").collect():
        if t.isTemporary or not t.tableName.startswith(prefijo):
            continue
        d = spark.sql(f"DESCRIBE DETAIL {t.tableName}").first()
        filas.append((t.tableName, spark.table(t.tableName).count(), round((d["sizeInBytes"] or 0) / 1e6, 1)))
    return spark.createDataFrame(filas, "tabla string, filas long, mb double")

In [ ]:
ESQUEMA_PROBLEMAS = "fuente string, regla string, accion string, registro_id string, detalle string, n_filas long"


# Reemplaza las filas de calidad de una tabla y muestra el resumen.
def registrar_calidad(df_problemas, tabla, condicion=None):
    df = (df_problemas
          .withColumn("tabla", F.lit(tabla))
          .withColumn("fecha", F.current_timestamp())
          .select("fuente", "tabla", "regla", "accion", "registro_id", "detalle",
                  F.col("n_filas").cast("long").alias("n_filas"), "fecha"))
    filtro = f"tabla = '{tabla}'" + (f" AND {condicion}" if condicion else "")
    if spark.catalog.tableExists("silver_calidad_log"):
        df.write.format("delta").mode("overwrite").option("replaceWhere", filtro).saveAsTable("silver_calidad_log")
    else:
        df.write.format("delta").mode("overwrite").partitionBy("tabla").saveAsTable("silver_calidad_log")
    resumen = (spark.table("silver_calidad_log").where(F.col("tabla") == tabla)
               .groupBy("regla", "accion").agg(F.sum("n_filas").alias("filas")).orderBy(F.desc("filas")).collect())
    for r in resumen:
        print(f"  calidad_log [{tabla}] {r['accion']:<11} {r['regla']}: {r['filas']:,}")


# DataFrame de problemas de calidad a partir de una lista.
def problemas(filas):
    return spark.createDataFrame([(f["fuente"], f["regla"], f["accion"], f["registro_id"], f["detalle"], int(f["n_filas"])) for f in filas], ESQUEMA_PROBLEMAS)


# Hace MERGE por clave, o crea la tabla si no existe.
def merge_silver(df, tabla, claves, particion=None):
    if spark.catalog.tableExists(tabla):
        df.createOrReplaceTempView("nuevo")
        condicion = " AND ".join(f"d.{k} = n.{k}" for k in claves)
        spark.sql(f"MERGE INTO {tabla} d USING nuevo n ON {condicion} WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
    else:
        escribir(df, tabla, particion, contar=False)
    print(tabla, spark.table(tabla).count())

In [ ]:
VS30_RELLENO_MAR = 600.0


# Asigna a cada píxel de mar el Vs30 del píxel de tierra más cercano.
def rellenar_mar(pdf):
    malla = pdf.pivot_table(index="fila", columns="col", values="vs30_usgs").sort_index().sort_index(axis=1)
    v = malla.to_numpy(float)
    mar = np.isclose(v, VS30_RELLENO_MAR) | ~np.isfinite(v)
    v = v[tuple(ndimage.distance_transform_edt(mar, return_distances=False, return_indices=True))]
    filas, cols = np.meshgrid(malla.index.to_numpy(), malla.columns.to_numpy(), indexing="ij")
    return pd.DataFrame({"fila": filas.ravel().astype("int64"), "col": cols.ravel().astype("int64"), "vs30_corregido": v.ravel(), "relleno_mar": mar.ravel()})


# Guarda la malla corregida y devuelve también sus límites.
def preparar_vs30():
    px = spark.table("bronze_vs30_pixeles").select("fila", "col", "lon", "lat", F.col("vs30").alias("vs30_usgs"))
    corregido = px.withColumn("_g", F.lit(0)).groupBy("_g").applyInPandas(rellenar_mar, "fila long, col long, vs30_corregido double, relleno_mar boolean")
    escribir(px.join(corregido, ["fila", "col"], "left"), "silver_tmp_vs30", contar=False)
    tabla = spark.table("silver_tmp_vs30")
    return tabla, tabla.agg(F.min("fila").alias("f0"), F.max("fila").alias("f1"), F.min("col").alias("c0"), F.max("col").alias("c1"),
                            F.max("lat").alias("lat0"), F.min("lat").alias("lat1"), F.min("lon").alias("lon0"), F.max("lon").alias("lon1")).first()


# Interpola el Vs30 de cada punto con joins a las 4 esquinas de la malla.
def con_vs30(df, lon="lon", lat="lat", salida="vs30"):
    g = malla_vs30
    u = (F.col(lon) - g.lon0) / ((g.lon1 - g.lon0) / (g.c1 - g.c0))
    v = (g.lat0 - F.col(lat)) / ((g.lat0 - g.lat1) / (g.f1 - g.f0))
    base = (df.withColumn("_u", u).withColumn("_v", v)
            .withColumn("_j", F.least(F.greatest(F.floor("_u"), F.lit(0)), F.lit(g.c1 - g.c0 - 1)).cast("long"))
            .withColumn("_i", F.least(F.greatest(F.floor("_v"), F.lit(0)), F.lit(g.f1 - g.f0 - 1)).cast("long")))
    for di, dj in ((0, 0), (0, 1), (1, 0), (1, 1)):
        esquina = vs30_px.select((F.col("fila") - g.f0 - di).alias("_i"), (F.col("col") - g.c0 - dj).alias("_j"), F.col("vs30_corregido").alias(f"_v{di}{dj}"))
        base = base.join(F.broadcast(esquina), ["_i", "_j"], "left")
    tx, ty = F.col("_u") - F.col("_j"), F.col("_v") - F.col("_i")
    valor = ((1 - tx) * (1 - ty) * F.col("_v00") + tx * (1 - ty) * F.col("_v01") + (1 - tx) * ty * F.col("_v10") + tx * ty * F.col("_v11"))
    return base.withColumn(salida, valor).drop("_u", "_v", "_i", "_j", "_v00", "_v01", "_v10", "_v11")

In [ ]:
try:
    spark.sql("SELECT st_contains(st_geomfromwkt('POLYGON((0 0,1 0,1 1,0 1,0 0))', 4326), st_point(0.5, 0.5, 4326)) AS ok").first()
    USAR_ST = True
except Exception:
    USAR_ST = False


# Caja envolvente de cada geometría WKT.
@F.pandas_udf("xmin double, ymin double, xmax double, ymax double")
def limites_wkt(wkt: pd.Series) -> pd.DataFrame:
    return pd.DataFrame(shapely.bounds(shapely.from_wkt(wkt.to_numpy(dtype=object))), columns=["xmin", "ymin", "xmax", "ymax"], index=wkt.index)


# Pasa lon y lat a UTM 18S.
@F.pandas_udf("x double, y double")
def utm_lima(lon: pd.Series, lat: pd.Series) -> pd.DataFrame:
    x, y = Transformer.from_crs(4326, EPSG_UTM_LIMA, always_xy=True).transform(lon.to_numpy(float), lat.to_numpy(float))
    return pd.DataFrame({"x": x, "y": y}, index=lon.index)


# Join espacial punto en polígono con funciones ST y filtro por caja.
def unir_poligonos(puntos, poligonos, atributos, clave, lon="lon", lat="lat"):
    pol = (poligonos.where(F.col("geometry_wkt").isNotNull()).select(*atributos, F.col("geometry_wkt").alias("_pol_wkt"))
           .withColumn("_pid", F.row_number().over(Window.orderBy(*atributos))))
    pts = puntos.select(*clave, F.col(lon).alias("_lon"), F.col(lat).alias("_lat"))
    if USAR_ST:
        pol = (pol.withColumn("_b", limites_wkt("_pol_wkt"))
               .select("*", *[F.col(f"_b.{c}").alias(f"_{c}") for c in ("xmin", "ymin", "xmax", "ymax")]).drop("_b"))
        cond = (F.col("_lon").between(F.col("_xmin"), F.col("_xmax")) & F.col("_lat").between(F.col("_ymin"), F.col("_ymax"))
                & F.expr("st_contains(st_geomfromwkt(_pol_wkt, 4326), st_point(_lon, _lat, 4326))"))
        cruces = pts.join(F.broadcast(pol), cond, "inner").select(*clave, "_pid", *atributos)
    else:
        filas = pol.collect()
        arbol = shapely.STRtree(shapely.from_wkt([f["_pol_wkt"] for f in filas]))
        pids = np.array([f["_pid"] for f in filas])
        valores = {c: [f[c] for f in filas] for c in atributos}
        esquema = T.StructType(list(pts.select(*clave).schema) + [T.StructField("_pid", T.IntegerType())] + list(pol.select(*atributos).schema))

        # Cruza puntos y polígonos con un índice espacial cuando no hay funciones ST.
        def cruzar(lotes):
            for pdf in lotes:
                i_pt, i_pol = arbol.query(shapely.points(pdf["_lon"].to_numpy(float), pdf["_lat"].to_numpy(float)), predicate="within")
                cruce = pdf.iloc[i_pt][clave].reset_index(drop=True)
                cruce["_pid"] = pids[i_pol]
                for c in atributos:
                    cruce[c] = [valores[c][k] for k in i_pol]
                yield cruce

        cruces = pts.mapInPandas(cruzar, esquema)
    primero = Window.partitionBy(*clave).orderBy("_pid")
    cruces = cruces.withColumn("_r", F.row_number().over(primero)).where("_r = 1").drop("_r", "_pid")
    return puntos.join(cruces, clave, "left")

In [ ]:
_CACHE_SDOF = {}


# Filtro discreto de un oscilador de un grado de libertad; guarda en caché cada periodo.
def _filtro_sdof(periodo, dt, z):
    if (periodo, dt, z) in _CACHE_SDOF:
        return _CACHE_SDOF[(periodo, dt, z)]
    w = 2 * np.pi / periodo
    A = np.array([[0.0, 1.0], [-w * w, -2 * z * w]])
    B = np.array([[0.0], [-1.0]])
    C = np.array([[1.0, 0.0]])
    D = np.array([[0.0]])
    Ad, Bd, Cd, Dd, _ = cont2discrete((A, B, C, D), dt, method="foh")
    b, a = ss2tf(Ad, Bd, Cd, Dd)
    _CACHE_SDOF[(periodo, dt, z)] = (b[0], a, w)
    return _CACHE_SDOF[(periodo, dt, z)]


# Espectro de respuesta Sa con 5 % de amortiguamiento.
def espectro_sa(acc_g, dt, periodos=PERIODOS, z=AMORTIGUAMIENTO):
    sa = np.empty(len(periodos))
    for k, periodo in enumerate(periodos):
        b, a, w = _filtro_sdof(float(periodo), float(dt), z)
        sa[k] = w * w * np.abs(lfilter(b, a, acc_g)).max()
    return sa


# Quita tendencia, aplica ventana y filtro pasa banda a una aceleración.
def filtrar_cms2(acc_cms2, fs):
    a = detrend(acc_cms2 - acc_cms2.mean(), type="linear")
    a = a * tukey(len(a), 0.05)
    sos = butter(4, FILTRO_HZ, btype="bandpass", fs=fs, output="sos")
    return sosfiltfilt(sos, a)


# Filtra una componente y calcula PGA, PGV y espectro.
def procesar_componente(acc_cms2, fs):
    dt = 1.0 / fs
    a = filtrar_cms2(acc_cms2, fs)
    vel = cumulative_trapezoid(a, dx=dt, initial=0.0)
    acc_g = a / G_CM_S2
    return acc_g, np.abs(acc_g).max(), np.abs(vel).max(), espectro_sa(acc_g, dt)


# Intensidad de Arias acumulada.
def intensidad_arias(acc_g, fs):
    return np.pi / (2 * 9.80665) * np.cumsum((acc_g * 9.80665) ** 2) / fs


# Duración significativa entre el 5 % y el 95 % de la intensidad de Arias.
def duracion_d595(arias, fs):
    return (np.searchsorted(arias, 0.95 * arias[-1]) - np.searchsorted(arias, 0.05 * arias[-1])) / fs


# PGA, PGV, espectro, Arias y duración de dos componentes horizontales.
def variables_onda(x1, x2, fs):
    k = min(len(x1), len(x2))
    pgas, pgvs, sas, arias, d595 = [], [], [], [], []
    for x in (x1[:k], x2[:k]):
        acc_g, pga, pgv, sa = procesar_componente(np.asarray(x, float), fs)
        ia = intensidad_arias(acc_g, fs)
        pgas.append(pga); pgvs.append(pgv); sas.append(sa); arias.append(ia[-1]); d595.append(duracion_d595(ia, fs))
    sa_geo = np.sqrt(sas[0] * sas[1])
    return {"pga_geo_g": float(np.sqrt(pgas[0] * pgas[1])), "pgv_geo_cms": float(np.sqrt(pgvs[0] * pgvs[1])),
            "tp_s": float(PERIODOS[int(sa_geo.argmax())]), "sa_max_geo_g": float(sa_geo.max()), "sa_geo": sa_geo.tolist(),
            "arias_ms": float(np.sqrt(arias[0] * arias[1])), "d595_s": float(np.sqrt(d595[0] * d595[1]))}

## 4. Sismos a procesar

In [ ]:
# Sismos de bronze dentro del rango de fechas.
MAESTROS = Path(f"{volumen}/landing/maestros")
eventos = sorted(r[0] for r in spark.table("bronze_cismid_registros").select("_evento_carga").distinct().collect()
                 if fecha_desde <= r[0][:10] <= fecha_hasta)
condicion_eventos = "evento IN (" + (", ".join(f"'{e}'" for e in eventos) or "''") + ")"
condicion_calidad = "(" + (" OR ".join(f"registro_id LIKE '{e}/%'" for e in eventos) or "FALSE") + ")"
print("Sismos a procesar:", eventos)

## 5. Acelerogramas: lectura de cabeceras y procesamiento de la señal

In [ ]:
SECCIONES = {
    "estacion": "INFORMATION ABOUT THE SEISMIC STATION",
    "sismo": "INFORMATION ABOUT THE EARTHQUAKE",
    "registro": "INFORMATION ABOUT THE RECORD",
    "datos": "ACCELERATION DATA",
}
NOMBRE_COMPONENTE = {"EO": "EW", "EW": "EW", "E": "EW", "NS": "NS", "N": "NS", "UD": "UD", "Z": "UD", "V": "UD"}


# Primer número que aparece en un texto.
def _num(texto):
    m = re.search(r"-?\d+(?:\.\d+)?", texto or "")
    return float(m.group()) if m else None


# Lee la cabecera y las tres componentes de un archivo CISMID.
def parsear_cismid(texto):
    lineas = [re.sub(r"^#\s?", "", l).rstrip() for l in texto.splitlines()]
    idx = {k: next(i for i, l in enumerate(lineas) if v in l.upper()) for k, v in SECCIONES.items()}

    # Lee los pares clave: valor entre dos líneas de la cabecera.
    def campos(a, b):
        d = {}
        for l in lineas[a + 1:b]:
            if ":" in l:
                k, v = l.split(":", 1)
                d.setdefault(re.sub(r"\s+", " ", k.strip()).upper(), v.strip())
        return d

    est = campos(idx["estacion"], idx["sismo"])
    sis = campos(idx["sismo"], idx["registro"])
    reg = campos(idx["registro"], idx["datos"])

    if "STATION CODE" in est:
        codigo, nombre = est["STATION CODE"].split()[0], est.get("STATION NAME", "").strip("﻿ ")
    else:
        st = est.get("STATION", "")
        codigo = st.split()[0] if st else None
        nombre = re.search(r"\((.*)\)", st).group(1) if "(" in st else st
    if "COORDINATES" in est:
        lat, lon = [float(x) for x in est["COORDINATES"].split(",")[:2]]
    else:
        lat, lon = _num(est.get("LATITUDE")), _num(est.get("LONGITUDE"))

    fecha_txt = sis.get("DATE (LOCAL)") or sis.get("DATE", "")
    try:
        fecha = datetime.strptime(fecha_txt.strip(), "%Y-%m-%d").date().isoformat()
    except ValueError:
        fecha = datetime.strptime(re.sub(r"\s+", " ", fecha_txt.strip()), "%B %d, %Y").date().isoformat()
    mag_txt = sis.get("MAGNITUDE", "")
    tipo_mag = re.search(r"\b(Mw|ML|Ms|mb|M)\b", mag_txt, re.I)
    max_txt = reg.get("MAXIMUM ACCELERATION") or reg.get("MAX. ACC. (FILTERED)") or ""
    max_cab = [float(x) for x in re.findall(r"-?\d+\.?\d*", re.sub(r"\([^)]*\)", "", max_txt))]

    cab = {
        "estacion": codigo, "estacion_nombre": nombre, "red": est.get("NETWORK"),
        "lat": lat, "lon": lon,
        "fs": _num(est.get("SAMPLING FREQUENCY (HZ)")),
        "fecha_evento": fecha,
        "hora_local": (sis.get("ORIGIN TIME (LOCAL)") or "").split()[0] if sis.get("ORIGIN TIME (LOCAL)") else None,
        "ev_lat": _num(sis.get("LATITUDE")), "ev_lon": _num(sis.get("LONGITUDE")),
        "ev_prof_km": _num(sis.get("DEPTH (KM)")),
        "magnitud": _num(mag_txt), "magnitud_tipo": tipo_mag.group(1) if tipo_mag else None,
        "fuente_sismo": sis.get("INFORMATION SOURCE"),
        "inicio_registro": reg.get("RECORD TIME (UTC-0)") or reg.get("START_TIME (UTC-0)") or reg.get("RECORD TIME (LOCAL)"),
        "n_cabecera": int(_num(reg.get("NUMBER OF SAMPLES") or reg.get("NUMBER DATA")) or 0),
        "unidades": reg.get("DATA UNITS"),
        "pga_cabecera_cms2": max(abs(x) for x in max_cab) if max_cab else None,
    }

    nombres = lineas[idx["datos"] + 1].split()
    filas = [l for l in lineas[idx["datos"] + 2:] if l.strip()]
    datos = pd.DataFrame(np.array([l.split() for l in filas], dtype=float), columns=nombres)
    datos = datos.rename(columns=lambda c: NOMBRE_COMPONENTE.get(c.upper(), c.upper()))
    return cab, datos[["EW", "NS", "UD"]]


# Valida un acelerograma y calcula sus variables; anota los problemas que encuentra.
def procesar_registro(evento, archivo, texto):
    fila = {"evento": evento, "archivo_estacion": archivo, "problemas": "", "procesado": False}
    avisos = []
    try:
        cab, datos = parsear_cismid(texto)
    except Exception as e:
        fila["problemas"] = f"cabecera_ilegible: {type(e).__name__}"
        return fila
    fila.update(cab)
    n = len(datos)
    fila["n_muestras"] = n
    if not cab["fs"]:
        fila["problemas"] = "sin_frecuencia_muestreo"
        return fila
    fila["duracion_s"] = n / cab["fs"]
    if cab["n_cabecera"] and cab["n_cabecera"] != n:
        avisos.append(f"n_muestras {n} != cabecera {cab['n_cabecera']}")
    if (cab["unidades"] or "").replace(" ", "").lower() not in ("cm/s2", "cm/s^2", "gal"):
        avisos.append(f"unidades '{cab['unidades']}'")
    if cab["lat"] is None or cab["lon"] is None:
        avisos.append("sin_coordenadas_estacion")
    if datos.isna().any().any() or n < 10 * cab["fs"]:
        fila["problemas"] = "datos_incompletos_o_registro_menor_a_10s"
        return fila

    pga_crudo = (datos - datos.mean()).abs().max().max()
    if cab["pga_cabecera_cms2"]:
        fila["pga_cabecera_dif_pct"] = 100 * (pga_crudo - cab["pga_cabecera_cms2"]) / cab["pga_cabecera_cms2"]
        if abs(fila["pga_cabecera_dif_pct"]) > 10:
            avisos.append(f"PGA difiere {fila['pga_cabecera_dif_pct']:.0f}% de la cabecera")

    for comp in ["EW", "NS", "UD"]:
        _, pga, pgv, sa = procesar_componente(datos[comp].to_numpy(float), cab["fs"])
        fila[f"pga_{comp.lower()}_g"], fila[f"pgv_{comp.lower()}_cms"], fila[f"sa_{comp.lower()}"] = pga, pgv, sa.tolist()
    fila["pga_geo_g"] = float(np.sqrt(fila["pga_ew_g"] * fila["pga_ns_g"]))
    fila["pgv_geo_cms"] = float(np.sqrt(fila["pgv_ew_cms"] * fila["pgv_ns_cms"]))
    sa_geo = np.sqrt(np.array(fila["sa_ew"]) * np.array(fila["sa_ns"]))
    fila["sa_geo"] = sa_geo.tolist()
    fila["tp_s"] = float(PERIODOS[int(sa_geo.argmax())])
    fila["sa_max_geo_g"] = float(sa_geo.max())
    fila["problemas"] = "; ".join(avisos)
    fila["procesado"] = True
    return fila

In [ ]:
ESQUEMA_REGISTRO = T.StructType([
    T.StructField("evento", T.StringType()), T.StructField("archivo_estacion", T.StringType()),
    T.StructField("estacion", T.StringType()), T.StructField("estacion_nombre", T.StringType()),
    T.StructField("red", T.StringType()),
    T.StructField("lat", T.DoubleType()), T.StructField("lon", T.DoubleType()),
    T.StructField("fs", T.DoubleType()), T.StructField("n_muestras", T.LongType()),
    T.StructField("duracion_s", T.DoubleType()), T.StructField("unidades", T.StringType()),
    T.StructField("fecha_evento", T.StringType()), T.StructField("hora_local", T.StringType()),
    T.StructField("ev_lat", T.DoubleType()), T.StructField("ev_lon", T.DoubleType()),
    T.StructField("ev_prof_km", T.DoubleType()), T.StructField("magnitud", T.DoubleType()),
    T.StructField("magnitud_tipo", T.StringType()), T.StructField("fuente_sismo", T.StringType()),
    T.StructField("inicio_registro", T.StringType()),
    T.StructField("pga_cabecera_cms2", T.DoubleType()), T.StructField("pga_cabecera_dif_pct", T.DoubleType()),
    *[T.StructField(f"pga_{c}_g", T.DoubleType()) for c in ["ew", "ns", "ud", "geo"]],
    *[T.StructField(f"pgv_{c}_cms", T.DoubleType()) for c in ["ew", "ns", "ud", "geo"]],
    T.StructField("tp_s", T.DoubleType()), T.StructField("sa_max_geo_g", T.DoubleType()),
    *[T.StructField(f"sa_{c}", T.ArrayType(T.DoubleType())) for c in ["ew", "ns", "ud", "geo"]],
    T.StructField("problemas", T.StringType()), T.StructField("procesado", T.BooleanType()),
])
COLUMNAS = [f.name for f in ESQUEMA_REGISTRO.fields]


# Procesa en los workers los registros de cada lote.
def procesar_lote(lotes):
    for pdf in lotes:
        reg = pd.DataFrame([procesar_registro(r.evento, r.archivo_estacion, r.contenido) for r in pdf.itertuples()]).reindex(columns=COLUMNAS)
        reg["n_muestras"] = reg["n_muestras"].astype("Int64")
        for c in [f.name for f in ESQUEMA_REGISTRO.fields if isinstance(f.dataType, (T.ArrayType, T.StringType))]:
            reg[c] = reg[c].astype(object).where(reg[c].notna(), None)
        yield reg


lote_registros = (spark.table("bronze_cismid_registros").where(F.col("_evento_carga").isin(eventos))
                  .select("evento", "archivo_estacion", "contenido")
                  .repartition(max(1, 8 * len(eventos))).mapInPandas(procesar_lote, ESQUEMA_REGISTRO))
escribir(lote_registros, "silver_tmp_ondas_registros")
registros = spark.table("silver_tmp_ondas_registros")
registros.groupBy("evento", "procesado").count().orderBy("evento").show(truncate=False)

## 6. Vs30 de cada estación y distancia al sismo

In [ ]:
# Deja la malla de Vs30 lista para los joins.
vs30_px, malla_vs30 = preparar_vs30()
print(f"Malla Vs30: filas {malla_vs30.f0}-{malla_vs30.f1} | columnas {malla_vs30.c0}-{malla_vs30.c1}")

In [ ]:
# Vs30 de cada estación y distancia al sismo.
R_TIERRA = 6371.0
lat1, lat2 = F.radians("lat"), F.radians("ev_lat")
dist_epi = 2 * R_TIERRA * F.asin(F.sqrt(F.sin((lat2 - lat1) / 2) ** 2
                                         + F.cos(lat1) * F.cos(lat2) * F.sin((F.radians("ev_lon") - F.radians("lon")) / 2) ** 2))

registros_vs = (con_vs30(registros)
    .withColumn("vs30", F.nanvl("vs30", F.lit(None).cast("double")))
    .withColumn("dist_epi_km", dist_epi)
    .withColumn("dist_hipo_km", F.sqrt(F.col("dist_epi_km") ** 2 + F.col("ev_prof_km") ** 2))
    .withColumn("periodos_s", F.array(*[F.lit(float(p)) for p in PERIODOS])))

## 7. MERGE de registros, cuarentena y registro de calidad

In [ ]:
# MERGE de registros válidos; cuarentena y calidad de los rechazados.
validos = (registros_vs.where("procesado").drop("procesado")
           .withColumn("problemas", F.when(F.col("vs30").isNull(), F.concat_ws("; ", F.col("problemas"), F.lit("estacion_sin_vs30")))
                                     .otherwise(F.col("problemas"))))
merge_silver(validos, "silver_ondas_registros", ["evento", "archivo_estacion"], particion="evento")

rechazados = registros_vs.where("NOT procesado").select("evento", "archivo_estacion", "estacion", F.col("problemas").alias("motivo_rechazo"))
if spark.catalog.tableExists("silver_ondas_registros_cuarentena"):
    rechazados.write.mode("overwrite").option("replaceWhere", condicion_eventos).saveAsTable("silver_ondas_registros_cuarentena")
else:
    rechazados.write.saveAsTable("silver_ondas_registros_cuarentena")
print("silver_ondas_registros_cuarentena", rechazados.count())

calidad = (registros_vs
    .withColumn("problemas", F.when(F.col("vs30").isNull() & F.col("procesado"),
                                    F.concat_ws("; ", F.col("problemas"), F.lit("estacion_sin_vs30")))
                              .otherwise(F.col("problemas")))
    .where((~F.col("procesado")) | (F.col("problemas") != ""))
    .select(F.lit("cismid").alias("fuente"),
            F.when(F.col("procesado"), "advertencia_registro").otherwise("registro_no_procesable").alias("regla"),
            F.when(F.col("procesado"), "advertencia").otherwise("descartado").alias("accion"),
            F.concat_ws("/", "evento", "archivo_estacion").alias("registro_id"),
            F.col("problemas").alias("detalle"), F.lit(1).alias("n_filas")))
registrar_calidad(calidad, "ondas_registros", condicion_calidad)
spark.sql("DROP TABLE IF EXISTS silver_tmp_ondas_registros")

## 8. Espectros y eventos

In [ ]:
# Espectros por componente y resumen por sismo, con MERGE.
validos = spark.table("silver_ondas_registros").where(F.col("evento").isin(eventos))
partes = []
for comp in ["ew", "ns", "ud", "geo"]:
    partes.append(validos.select(
        "evento", "estacion", "archivo_estacion", "lat", "lon", "vs30", "dist_hipo_km", "periodos_s",
        F.lit(comp.upper()).alias("componente"),
        F.posexplode(f"sa_{comp}").alias("indice_periodo", "Sa")))
espectros = (partes[0].unionByName(partes[1]).unionByName(partes[2]).unionByName(partes[3])
             .select("evento", "estacion", "archivo_estacion", "componente", "lat", "lon", "vs30", "dist_hipo_km",
                     "indice_periodo", F.element_at("periodos_s", F.col("indice_periodo") + 1).alias("T"), "Sa"))
merge_silver(espectros, "silver_espectros", ["evento", "archivo_estacion", "componente", "indice_periodo"], particion="evento")

eventos_df = (validos.groupBy("evento").agg(
        F.first("fecha_evento").alias("fecha"), F.first("hora_local").alias("hora_local"),
        F.first("ev_lat").alias("lat"), F.first("ev_lon").alias("lon"), F.first("ev_prof_km").alias("profundidad_km"),
        F.first("magnitud").alias("magnitud_cabecera"), F.first("magnitud_tipo").alias("magnitud_tipo"),
        F.count("*").alias("n_estaciones"),
        F.max("pga_geo_g").alias("pga_geo_max_g"), F.expr("percentile_approx(pga_geo_g, 0.5)").alias("pga_geo_mediana_g"),
        F.min("dist_hipo_km").alias("dist_hipo_min_km"))
    .withColumn("magnitud_catalogo", F.regexp_extract("evento", r"_M[wl]?(\d+\.\d+)$", 1).cast("double")))
merge_silver(eventos_df, "silver_eventos", ["evento"])
display(spark.table("silver_eventos").orderBy("fecha"))

## 9. Maestros
Se procesan la primera vez o cuando `recargar_maestros = true`.

In [ ]:
# Los maestros se procesan la primera vez o si se pide recargarlos.
procesar_maestros = recargar_maestros or not spark.catalog.tableExists("silver_edificios")
print("Procesar maestros:", procesar_maestros)

### Territorio y edificios de los 6 distritos

In [ ]:
# Umbrales de calidad de edificios y de cálculo de pisos.
if procesar_maestros:
    CONFIANZA_MIN = 0.65
    ALTURA_POR_PISO_M = 3.0
    FRACCION_ALTURA_ESTABLE = 0.75
    PESO_REFERENCIA_WEB = 20
    ALTURA_CORTE_TORRE_M = 40.0
    VENTANA_VECINDARIO_PX = 15
    OSM_MIN_PISOS_CONTRASTE = 5
    OSM_PISOS_SATURACION = 18
    H3_RES_EDIFICIO = 9
    H3_RES_SUELO = 8

In [ ]:
if procesar_maestros:
    ESQ_DISTRITOS = ("ubigeo string, distrito string, provincia string, departamento string, area_km2 double, centroide_lon double, centroide_lat double, "
                     "geometry_wkt string, geometria_fuente string, en_estudio boolean, geometria_reparada boolean, area_recortada_km2 double")


    # Provincia según los dos primeros dígitos del ubigeo.
    def provincia_de(columna):
        return F.when(F.substring(columna, 1, 2) == "15", "LIMA").when(F.substring(columna, 1, 2) == "07", "CALLAO")


    # Repara geometrías, recorta vecinos y calcula área y centroide de cada distrito.
    def geometria_distritos(lim):
        gdf = gpd.GeoDataFrame(lim, geometry=gpd.GeoSeries.from_wkt(lim["geometry_wkt"].astype(object)), crs=4326)
        reparada = (~gdf.is_valid).to_numpy()
        gdf["geometry"] = gdf.geometry.make_valid()
        area_antes = gdf.to_crs(EPSG_UTM_LIMA).area
        completados = gdf["geometria_fuente"] != "INEI"
        if completados.any() and not completados.all():
            gdf.loc[~completados, "geometry"] = gdf.loc[~completados].geometry.difference(gdf.loc[completados].geometry.union_all())
        utm = gdf.to_crs(EPSG_UTM_LIMA)
        centroides = utm.centroid.to_crs(4326)
        return pd.DataFrame({"ubigeo": gdf["IDDIST"], "distrito": gdf["NOMBDIST"], "provincia": gdf["NOMBPROV"], "departamento": gdf["NOMBDEP"],
                             "area_km2": (utm.area / 1e6).round(3), "centroide_lon": centroides.x, "centroide_lat": centroides.y,
                             "geometry_wkt": gdf.geometry.to_wkt(), "geometria_fuente": gdf["geometria_fuente"], "en_estudio": gdf["IDDIST"].isin(list(DISTRITOS_ESTUDIO)),
                             "geometria_reparada": reparada, "area_recortada_km2": ((area_antes - utm.area) / 1e6).round(3)})


    lim = (spark.table("bronze_limites_inei").where(F.col("NOMBPROV").isin(PROVINCIAS_ESTUDIO))
           .select("IDDIST", "NOMBDIST", "NOMBPROV", "NOMBDEP", "geometry_wkt", F.lit("INEI").alias("geometria_fuente")))
    if spark.catalog.tableExists("bronze_limites_osm_distritos"):
        inei = lim.select(F.col("IDDIST").alias("ubigeo"), F.col("NOMBPROV").alias("_prov"), F.col("NOMBDEP").alias("_dep"))
        sin_tilde = F.upper(F.translate(F.col("nombre"), "ÁÉÍÓÚÜÑáéíóúüñ-", "AEIOUUNaeiouun "))
        lim = (spark.table("bronze_limites_osm_distritos").join(inei, "ubigeo", "left")
               .select(F.col("ubigeo").alias("IDDIST"), sin_tilde.alias("NOMBDIST"), F.coalesce("_prov", provincia_de("ubigeo")).alias("NOMBPROV"),
                       F.coalesce("_dep", provincia_de("ubigeo")).alias("NOMBDEP"), "geometry_wkt", F.lit("OSM").alias("geometria_fuente")))
    elif spark.catalog.tableExists("bronze_limites_complemento_osm"):
        comp = spark.table("bronze_limites_complemento_osm").select("IDDIST", F.col("geometry_wkt").alias("_wkt"), F.col("fuente").alias("_fuente_c"))
        falta = F.col("geometry_wkt").isNull() & F.col("_wkt").isNotNull()
        lim = (lim.join(comp, "IDDIST", "left")
               .withColumn("geometria_fuente", F.when(falta, F.col("_fuente_c")).otherwise(F.col("geometria_fuente")))
               .withColumn("geometry_wkt", F.when(falta, F.col("_wkt")).otherwise(F.col("geometry_wkt")))
               .drop("_wkt", "_fuente_c"))
    sin_geometria = [r.NOMBDIST for r in lim.where("geometry_wkt IS NULL").select("NOMBDIST").collect()]
    escribir(lim.where("geometry_wkt IS NOT NULL").withColumn("_g", F.lit(0)).groupBy("_g").applyInPandas(geometria_distritos, ESQ_DISTRITOS), "silver_distritos")

    dist = spark.table("silver_distritos")
    print("Por provincia:", {r["provincia"]: r["count"] for r in dist.groupBy("provincia").count().collect()}, "| geometrías reparadas:", dist.where("geometria_reparada").count(),
          "| sin geometría:", sin_geometria, "| recortes a vecinos (km²):", {r.distrito: r.area_recortada_km2 for r in dist.where("area_recortada_km2 > 0.01").collect()})
    estudio = dist.where("en_estudio")
    faltan_estudio = set(DISTRITOS_ESTUDIO) - {r.ubigeo for r in estudio.select("ubigeo").collect()}
    assert not faltan_estudio, f"Distritos de estudio sin geometría: {faltan_estudio}"
    print("Área de estudio:", [r.distrito for r in estudio.select("distrito").collect()], f"| {estudio.agg(F.sum('area_km2')).first()[0]:.1f} km²")

    BBOX = np.array(estudio.select(limites_wkt("geometry_wkt").alias("b")).agg(F.min("b.xmin"), F.min("b.ymin"), F.max("b.xmax"), F.max("b.ymax")).first())
    print("BBOX del área de estudio:", BBOX.round(3))


    # Agrega ubigeo, distrito y provincia a cada punto.
    def asignar_distrito(df, clave):
        return unir_poligonos(df, spark.table("silver_distritos"), ["ubigeo", "distrito", "provincia"], clave)

In [ ]:
if procesar_maestros:
    perfiles = spark.table("bronze_e030_perfiles").drop("_fuente", "_source_file", "_ingested_at")
    factor_s = spark.table("bronze_e030_factor_suelo").where(F.col("zona") == ZONA_E030_LIMA).select("perfil", "S")
    z = spark.table("bronze_e030_zonas").where(F.col("zona") == ZONA_E030_LIMA).first()["Z_g"]
    e030 = (perfiles.join(factor_s, "perfil").withColumn("zona", F.lit(ZONA_E030_LIMA)).withColumn("Z_g", F.lit(z))
            .select("perfil", "descripcion", "vs30_min", "vs30_max", "zona", "Z_g", "S", "TP_s", "TL_s").orderBy("perfil"))
    escribir(e030, "silver_e030_parametros")
    e030 = spark.table("silver_e030_parametros")
    E030 = {r["perfil"]: r.asDict() for r in e030.collect()}
    e030.orderBy("perfil").show(truncate=False)


    # Perfil de suelo E.030 según el Vs30.
    def perfil_e030(col_vs30):
        return (F.when(col_vs30 > E030["S0"]["vs30_min"], "S0")
                 .when(col_vs30 > E030["S1"]["vs30_min"], "S1")
                 .when(col_vs30 >= E030["S2"]["vs30_min"], "S2")
                 .when(col_vs30.isNotNull(), "S3"))

In [ ]:
# Suelo: distrito, perfil E.030 y H3 de cada píxel de Vs30.
if procesar_maestros:
    vs = vs30_px.select("fila", "col", "lon", "lat", "vs30_usgs", F.col("vs30_corregido").alias("vs30"), "relleno_mar")
    vs_dist = asignar_distrito(vs, ["fila", "col"])

    e030_b = F.broadcast(e030.select("perfil", "S", "TP_s", "TL_s", "Z_g"))
    suelo = (vs_dist
             .withColumn("perfil", perfil_e030(F.col("vs30")))
             .join(e030_b, "perfil", "left")
             .withColumn("en_area_estudio", F.col("ubigeo").isin(list(DISTRITOS_ESTUDIO)))
             .withColumn("h3_r8", F.expr(f"h3_longlatash3string(lon, lat, {H3_RES_SUELO})"))
             .withColumnRenamed("TP_s", "TP").withColumnRenamed("TL_s", "TL")
             .select("fila", "col", "lon", "lat", "vs30", "vs30_usgs", "relleno_mar", "perfil", "S", "TP", "TL", "Z_g",
                     "ubigeo", "distrito", "provincia", "en_area_estudio", "h3_r8"))
    escribir(suelo, "silver_suelo")

    s = spark.table("silver_suelo")
    s.where("en_area_estudio").groupBy("perfil").agg(F.count("*").alias("pixeles"), F.round(F.avg("vs30")).alias("vs30_medio")).orderBy("perfil").show()
    registrar_calidad(problemas([
        {"fuente": "vs30_usgs", "regla": "pixel_fuera_del_area_estudio", "accion": "fuera_de_alcance", "registro_id": None,
         "detalle": "píxel del rectángulo cuyo centro no cae en los distritos de estudio; se conserva para interpolar",
         "n_filas": s.where("NOT en_area_estudio").count()},
        {"fuente": "vs30_usgs", "regla": "vs30_relleno_mar_600", "accion": "imputado", "registro_id": None,
         "detalle": "píxel con el relleno de mar de USGS (600 m/s): toma el Vs30 del píxel de tierra más cercano",
         "n_filas": s.where("relleno_mar").count()},
        {"fuente": "vs30_usgs", "regla": "vs30_nulo", "accion": "descartado", "registro_id": None,
         "detalle": "píxel sin valor de Vs30", "n_filas": s.where("vs30_usgs IS NULL").count()},
    ]), "suelo")

In [ ]:
if procesar_maestros:
    DISTRITO_PDF = {"cercado_de_lima": "LIMA", "comas": "COMAS", "la_molina": "LA MOLINA", "villa_el_salvador": "VILLA EL SALVADOR"}
    ROMANOS = ["I", "II", "III", "IV", "V"]
    ESQ_ZONAS = ("archivo_pdf string, distrito string, zona string, descripcion string, perfiles_e030 array<string>, "
                 "periodos_s array<double>, menciona_licuacion boolean, menciona_relleno boolean")
    ESQ_PERIODOS = "archivo_pdf string, distrito string, pagina long, T_min_s double, T_max_s double, contexto string"


    # Une espacios y arregla números cortados en el texto de un PDF.
    def normalizar(texto):
        texto = re.sub(r"\s+", " ", texto)
        return re.sub(r"(\d)\s*([.,])\s*(\d)", r"\1.\3", texto)


    # Rangos de periodo del suelo mencionados en un PDF de microzonificación.
    def periodos_pdf(d):
        archivo = d["distrito"].iloc[0]
        periodos = []
        for r in d.sort_values("pagina").itertuples():
            t = normalizar(r.texto)
            for m in re.finditer(r"(\d\.\d+)\s*(?:s|seg\.?)?\s*(?:y|a|–|-|hasta)\s*(\d\.\d+)\s*(?:s|seg)\b", t):
                tmin, tmax = float(m.group(1)), float(m.group(2))
                if 0 < tmin < tmax <= 5:
                    periodos.append({"archivo_pdf": archivo, "distrito": DISTRITO_PDF[archivo], "pagina": r.pagina,
                                     "T_min_s": tmin, "T_max_s": tmax, "contexto": t[max(0, m.start() - 200): m.end() + 60]})
        columnas = ["archivo_pdf", "distrito", "pagina", "T_min_s", "T_max_s", "contexto"]
        return pd.DataFrame(periodos, columns=columnas).drop_duplicates(["distrito", "pagina", "T_min_s", "T_max_s"])


    # Zonas de suelo descritas en un PDF de microzonificación.
    def zonas_pdf(d):
        archivo = d["distrito"].iloc[0]
        texto = normalizar(" ".join(d.sort_values("pagina")["texto"]))
        marcas = list(re.finditer(r"\bZona\s+(IV|V|I{1,3})\s*[:.\-–]?\s+(?=Esta|Comprende|Corresponde|Se |En |Está|Abarca|Conformad)", texto, re.I))
        zonas = []
        for zona in ROMANOS:
            m = next((m for m in marcas if m.group(1).upper() == zona), None)
            if not m:
                continue
            sig = next((n.start() for n in marcas if n.start() > m.start() and n.group(1).upper() != zona), m.end() + 2500)
            desc = texto[m.end(): min(sig, m.end() + 2500)].strip()
            zonas.append({"archivo_pdf": archivo, "distrito": DISTRITO_PDF[archivo], "zona": zona, "descripcion": desc,
                          "perfiles_e030": sorted(set(re.findall(r"\bS[0-4]\b", desc))),
                          "periodos_s": sorted({float(x.replace(",", ".")) for x in re.findall(r"(\d[.,]\d{1,2})\s*(?:s\b|seg)", desc)}),
                          "menciona_licuacion": bool(re.search(r"licuac", desc, re.I)),
                          "menciona_relleno": bool(re.search(r"relleno|desmonte", desc, re.I))})
        return pd.DataFrame(zonas, columns=["archivo_pdf", "distrito", "zona", "descripcion", "perfiles_e030", "periodos_s", "menciona_licuacion", "menciona_relleno"])


    pag = spark.table("bronze_microzonificacion_paginas").select("distrito", "pagina", "texto")
    escribir(pag.groupBy("distrito").applyInPandas(zonas_pdf, ESQ_ZONAS), "silver_microzonificacion_zonas")
    escribir(pag.groupBy("distrito").applyInPandas(periodos_pdf, ESQ_PERIODOS), "silver_microzonificacion_periodos")
    display(spark.table("silver_microzonificacion_zonas").select("distrito", "zona", "menciona_relleno", "menciona_licuacion", F.substring("descripcion", 1, 90).alias("descripcion")))
    display(spark.table("silver_microzonificacion_periodos").select("distrito", "pagina", "T_min_s", "T_max_s"))

In [ ]:
# Proporción de tipologías de vivienda por distrito.
if procesar_maestros:
    GRUPO_MATERIAL = {"Ladrillo o bloque de cemento": "ladrillo", "Piedra o sillar con cal o cemento": "ladrillo",
                      "Adobe": "adobe", "Tapia": "adobe", "Piedra con barro": "adobe", "Quincha (caña con barro)": "quincha",
                      "Madera (pona, tornillo etc.)": "madera", "Triplay / calamina / estera": "madera", "Otro material": "madera"}
    if spark.catalog.tableExists("bronze_censo2017_paredes"):
        grupo = F.create_map(*[x for par in GRUPO_MATERIAL.items() for x in (F.lit(par[0]), F.lit(par[1]))])
        por_grupo = (spark.table("bronze_censo2017_paredes").withColumn("grupo", grupo[F.col("material")])
                     .groupBy("ubigeo", "distrito").pivot("grupo", ["ladrillo", "adobe", "quincha", "madera"]).agg(F.sum("viviendas"))
                     .fillna(0))
        total = F.col("ladrillo") + F.col("adobe") + F.col("quincha") + F.col("madera")
        tipologia_censo = (por_grupo.withColumn("viviendas", total)
                           .select("ubigeo", "distrito", "viviendas",
                                   *[(F.col(g) / F.col("viviendas")).alias(f"p_{g}") for g in ["ladrillo", "adobe", "quincha", "madera"]]))
        escribir(tipologia_censo, "silver_tipologia_censo")
        faltan = spark.table("silver_distritos").join(spark.table("silver_tipologia_censo"), "ubigeo", "left_anti").select("ubigeo", "distrito").collect()
        print("Distritos de silver.distritos sin dato del censo (se tratan como albañilería):", [r.distrito for r in faltan])
        (spark.table("silver_tipologia_censo").orderBy(F.desc("p_adobe"))
         .select("distrito", "viviendas", *[F.round(100 * F.col(f"p_{g}"), 1).alias(f"pct_{g}") for g in ["ladrillo", "adobe", "quincha", "madera"]])
         .show(12, truncate=False))
    else:
        print("Falta bronze.censo2017_paredes (Parte 1, sección 13): gold usará solo albañilería y concreto")

In [ ]:
# Edificios de los 6 distritos; separa baja confianza y duplicados.
if procesar_maestros:
    ob = spark.table("bronze_open_buildings")
    n_bronze = ob.count()
    en_bbox = ob.where(F.col("longitude").between(float(BBOX[0]), float(BBOX[2])) & F.col("latitude").between(float(BBOX[1]), float(BBOX[3])))

    base = en_bbox.select(F.col("full_plus_code").alias("id"), F.col("longitude").alias("lon"), F.col("latitude").alias("lat"),
                          F.col("area_in_meters").alias("area_m2"), F.col("confidence").alias("confianza"),
                          F.col("geometry").alias("geometry_wkt"))
    escribir(asignar_distrito(base, ["id"]), "silver_tmp_edificios_distrito")
    con_dist = spark.table("silver_tmp_edificios_distrito")
    n_bbox = con_dist.count()

    lima = con_dist.where(F.col("ubigeo").isin(list(DISTRITOS_ESTUDIO)))
    n_lima = lima.count()
    baja_conf = lima.where(F.col("confianza") < CONFIANZA_MIN)
    edif = lima.where(F.col("confianza") >= CONFIANZA_MIN)
    n_dups = edif.groupBy("id").count().where("count > 1").count()
    edif = (edif.withColumn("_r", F.row_number().over(Window.partitionBy("id").orderBy(F.desc("confianza"), "lon", "lat")))
                .where("_r = 1").drop("_r"))
    print(f"bronze: {n_bronze:,} | en rectángulo: {n_bbox:,} | en los distritos de estudio: {n_lima:,} | "
          f"confianza ≥ {CONFIANZA_MIN}: {n_lima - baja_conf.count():,} | ids duplicados: {n_dups}")

In [ ]:
# Hexágonos H3 de cada edificio.
if procesar_maestros:
    edif = (edif
            .withColumn("h3_id", F.expr(f"h3_longlatash3string(lon, lat, {H3_RES_EDIFICIO})"))
            .withColumn("h3_r8", F.expr(f"h3_toparent(h3_id, {H3_RES_SUELO})")))

In [ ]:
if procesar_maestros:
    teselas = (spark.table("bronze_open_buildings_25d_teselas").select("tesela", "anio", "xmin", "ymin", "xmax", "ymax", "res_m", "nodata")
               .withColumn("ruta", F.concat(F.lit((MAESTROS / "open_buildings_25d").as_posix() + "/"), "tesela", F.lit(".tif"))))
    ANIOS_25D = sorted(int(r.anio) for r in teselas.select("anio").distinct().collect())
    print(f"Teselas de altura: {teselas.count()} | años: {ANIOS_25D}")

    huella = (edif.select("id", "lon", "lat", "area_m2", "geometry_wkt").withColumn("_xy", utm_lima("lon", "lat"))
              .withColumn("x_utm", F.col("_xy.x")).withColumn("y_utm", F.col("_xy.y")).drop("_xy"))
    escribir(huella, "silver_tmp_edificios_huella")
    huella = spark.table("silver_tmp_edificios_huella")


    # Convierte huellas WKT a geometrías en UTM.
    def huellas_utm(wkt):
        transformador = Transformer.from_crs(4326, EPSG_UTM_LIMA, always_xy=True)
        return shapely.transform(shapely.from_wkt(wkt.astype(object).where(wkt.notna(), None).to_numpy(dtype=object)),
                                 lambda c: np.column_stack(transformador.transform(c[:, 0], c[:, 1])))


    # Percentiles de altura de los píxeles de cada huella.
    def percentiles_huella(h, etiquetas, qs):
        m = etiquetas > 0
        lab, val = etiquetas[m], h[m]
        orden = np.lexsort((val, lab))
        lab, val = lab[orden], val[orden]
        presentes, inicio, n = np.unique(lab, return_index=True, return_counts=True)
        pos_q = inicio[:, None] + np.asarray(qs)[None, :] * (n[:, None] - 1)
        bajo = np.floor(pos_q).astype(int)
        alto = np.minimum(bajo + 1, (inicio + n - 1)[:, None])
        return presentes, val[bajo] + (val[alto] - val[bajo]) * (pos_q - bajo)


    # Se ejecuta por tesela: rasteriza las huellas y mide sus percentiles de altura.
    def alturas_tesela(d):
        d = d.sort_values("id").reset_index(drop=True)
        t = d.iloc[0]
        with rasterio.open(t["ruta"]) as src:
            h = src.read(1).astype("float32")
            transform_t = src.transform
        h[(h == t["nodata"]) | ~np.isfinite(h)] = 0.0
        etiquetas = rasterize(zip(huellas_utm(d["geometry_wkt"]), np.arange(1, len(d) + 1)), out_shape=h.shape, transform=transform_t, fill=0, dtype="int32")
        etiquetas[h <= 0] = 0
        presentes, pct = percentiles_huella(h, etiquetas, [0.5, 0.9, 1.0])
        salida = pd.DataFrame({"id": d["id"].to_numpy(), "anio": int(t["anio"]), "tesela": t["tesela"],
                               "p50": np.nan, "p90": np.nan, "pmax": np.nan, "centro": np.nan, "vecindario": np.nan})
        salida.loc[presentes - 1, ["p50", "p90", "pmax"]] = pct
        if int(t["anio"]) == ANIOS_25D[-1]:
            colp = np.clip(((d["x_utm"].to_numpy() - t["xmin"]) // t["res_m"]).astype(int), 0, h.shape[1] - 1)
            filp = np.clip(((t["ymax"] - d["y_utm"].to_numpy()) // t["res_m"]).astype(int), 0, h.shape[0] - 1)
            salida["centro"] = h[filp, colp]
            suma = ndimage.uniform_filter(h, VENTANA_VECINDARIO_PX)[filp, colp]
            cubierto = ndimage.uniform_filter((h > 0).astype("float32"), VENTANA_VECINDARIO_PX)[filp, colp]
            salida["vecindario"] = np.where(cubierto > 0, suma / np.maximum(cubierto, 1e-6), np.nan)
        return salida


    en_tesela = ((F.col("x_utm") >= F.col("xmin")) & (F.col("x_utm") < F.col("xmax")) & (F.col("y_utm") > F.col("ymin")) & (F.col("y_utm") <= F.col("ymax")))
    alturas = (huella.where(F.col("geometry_wkt").isNotNull()).join(F.broadcast(teselas), en_tesela)
               .groupBy("tesela").applyInPandas(alturas_tesela, "id string, anio long, tesela string, p50 double, p90 double, pmax double, centro double, vecindario double"))
    escribir(alturas, "silver_tmp_alturas_tesela")

In [ ]:
if procesar_maestros:
    if spark.catalog.tableExists("bronze_osm_niveles"):
        osm_pt = spark.table("bronze_osm_niveles").where(F.col("niveles").between(1, 40)).select("lon", "lat", "niveles").collect()
    else:
        print("Sin bronze_osm_niveles: los pisos salen solo de la altura 2.5D")
        osm_pt = []
    osm_x, osm_y = Transformer.from_crs(4326, EPSG_UTM_LIMA, always_xy=True).transform([p.lon for p in osm_pt], [p.lat for p in osm_pt])
    osm_x, osm_y, osm_niv = np.asarray(osm_x, float), np.asarray(osm_y, float), np.array([float(p.niveles) for p in osm_pt])
    ruta_zonas = (MAESTROS / "microzonificacion_cismid" / "zonas_cismid_utm18s_25m.tif").as_posix()
    hay_zonas = Path(ruta_zonas).exists()
    if not hay_zonas:
        print("Falta", ruta_zonas, "(ver README): el perfil de suelo sale solo del Vs30")

    COLS_P50, COLS_P90 = [f"{a}_p50" for a in ANIOS_25D], [f"{a}_p90" for a in ANIOS_25D]
    ESQ_RASGOS = ("id string, lon double, lat double, area_m2 double, x_utm double, y_utm double, tesela string, altura_m double, altura_metodo string, "
                  "altura_ultimo_anio_m double, altura_p90_m double, altura_p90_ultimo_anio_m double, altura_vecindario_m double, n_anios_altura long, "
                  "altura_max_m double, niveles_osm double, zona_cismid long")
    COLS_RASGOS = [c.split()[0] for c in ESQ_RASGOS.split(", ")]


    # Mediana de los pisos OSM cuyo punto cae dentro de cada huella.
    def niveles_en_huella(pos):
        salida = np.full(len(pos), np.nan)
        if len(osm_niv):
            i_pt, i_h = shapely.STRtree(huellas_utm(pos["geometry_wkt"])).query(shapely.points(osm_x, osm_y), predicate="within")
            if len(i_pt):
                mediana = pd.Series(osm_niv[i_pt]).groupby(i_h).median()
                salida[mediana.index.to_numpy(int)] = mediana.to_numpy()
        return salida


    # Calcula altura multianual, zona CISMID y pisos OSM de cada lote de edificios.
    def rasgos_altura(lotes):
        zonas, tz = None, None
        if hay_zonas:
            with rasterio.open(ruta_zonas) as src_z:
                zonas, tz = src_z.read(1), src_z.transform
        for pos in lotes:
            serie = pd.DataFrame(pos[COLS_P50].to_numpy(float), columns=ANIOS_25D)
            reciente = serie.iloc[:, -2:].median(axis=1)
            existe = serie.ge(FRACCION_ALTURA_ESTABLE * reciente, axis=0)
            altura_estable = serie.where(existe).median(axis=1).to_numpy()
            centro, alt_max = pos["centro"].to_numpy(float), pos["pmax"].to_numpy(float)
            pos["altura_m"] = np.where(altura_estable > 0, altura_estable, np.where(centro > 0, centro, np.nan))
            pos["altura_metodo"] = np.where(altura_estable > 0, "huella_multianual", np.where(centro > 0, "centroide", "sin_altura"))
            pos["altura_ultimo_anio_m"] = serie.iloc[:, -1].to_numpy()
            pos["altura_p90_m"] = pd.DataFrame(pos[COLS_P90].to_numpy(float)).where(existe.to_numpy()).median(axis=1).to_numpy()
            pos["altura_p90_ultimo_anio_m"] = pos[COLS_P90[-1]].to_numpy(float)
            pos["altura_vecindario_m"] = pos["vecindario"].to_numpy(float)
            pos["n_anios_altura"] = existe.sum(axis=1).to_numpy()
            pos["altura_max_m"] = np.where(alt_max > 0, alt_max, pos["altura_m"])
            pos["niveles_osm"] = niveles_en_huella(pos)
            pos["zona_cismid"] = 0
            if zonas is not None:
                cz = np.floor((pos["x_utm"].to_numpy() - tz.c) / tz.a).astype(int)
                fz = np.floor((pos["y_utm"].to_numpy() - tz.f) / tz.e).astype(int)
                dentro = (cz >= 0) & (cz < zonas.shape[1]) & (fz >= 0) & (fz < zonas.shape[0])
                pos.loc[dentro, "zona_cismid"] = zonas[fz[dentro], cz[dentro]]
            yield pos[COLS_RASGOS]


    por_anio = spark.table("silver_tmp_alturas_tesela")
    ancho = por_anio.groupBy("id").pivot("anio", ANIOS_25D).agg(F.first("p50").alias("p50"), F.first("p90").alias("p90"))
    ultimo = por_anio.where(F.col("anio") == ANIOS_25D[-1]).select("id", "tesela", "pmax", "centro", "vecindario")
    escribir(con_vs30(huella.join(ancho, "id", "left").join(ultimo, "id", "left").mapInPandas(rasgos_altura, ESQ_RASGOS)), "silver_tmp_edificios_rasgos")

    rasgos = spark.table("silver_tmp_edificios_rasgos")
    rasgos.groupBy("altura_metodo").count().show()
    rasgos.where("altura_metodo = 'huella_multianual'").groupBy("n_anios_altura").count().orderBy("n_anios_altura").show()
    rasgos.groupBy("zona_cismid").count().orderBy("zona_cismid").show()
    print(f"Edificios con pisos de OSM (punto dentro de la huella): {rasgos.where('niveles_osm IS NOT NULL').count():,} de {len(osm_pt):,} puntos OSM")

In [ ]:
if procesar_maestros:
    REFERENCIA_WEB = [
        ("Torre Banco de la Nación", 30, -12.087128, -77.000533, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Edificio BBVA", 19, -12.093705, -77.021076, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Begonias", 26, -12.092114, -77.023963, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("The Westin Lima", 30, -12.091474, -77.024632, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Chocavento", 25, -12.097605, -77.020904, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Barlovento", 31, -12.097867, -77.020646, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Fórum", 30, -12.100471, -77.018711, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Panamá", 29, -12.098188, -77.019618, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Orquídeas", 27, -12.092008, -77.028473, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("T Tower", 25, -12.090926, -77.026784, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Javier Prado", 27, -12.092002, -77.028816, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("JW Marriott Lima", 25, -12.131516, -77.029504, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Lima Central Tower", 25, -12.097544, -76.972975, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Edificio Petroperú", 22, -12.097900, -77.024900, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Edificio Capital", 23, -12.092674, -77.026500, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Capital El Derby", 22, -12.097781, -76.972420, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Edificio Alto Caral", 21, -12.095917, -77.022019, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Scotiabank", 21, -12.095145, -77.021962, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Edificio Altavista", 22, -12.099811, -77.018983, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Edificio Beyond", 29, -12.097128, -76.973186, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Centro de Convenciones de Lima", 10, -12.086374, -77.000574, "wikidata.org/wiki/Q18643986"),
        ("Hyatt Centric San Isidro", 10, -12.094986, -77.036343, "hotels.com/ho636542528"),
        ("Courtyard Lima Miraflores", 11, -12.123411, -77.027565, "travelweekly.com/Hotels/Lima-Peru/Courtyard-by-Marriott-Lima-Miraflores-p9447790"),
        ("Casa Andina Premium Miraflores", 18, -12.122044, -77.026590, "hotels.com/ho268714"),
        ("Pullman Lima San Isidro", 9, -12.095171, -77.037879, "successfulmeetings.com (Pullman Lima San Isidro)"),
        ("Sonesta Hotel El Olivar", 7, -12.097809, -77.035351, "northstarmeetingsgroup.com (Sonesta Hotel El Olivar)"),
        ("Belmond Miraflores Park", 12, -12.135324, -77.027303, "meetings-conventions.com (Miraflores Park, A Belmond Hotel)"),
        ("Swissôtel Lima", 18, -12.097248, -77.038190, "atrapalo.com.ar/hoteles/42366-0_hotel-swissotel-lima"),
        ("Novotel Lima San Isidro", 8, -12.096003, -77.036521, "successfulmeetings.com (Novotel Lima)"),
        ("Hilton Garden Inn Lima Miraflores", 18, -12.123770, -77.032795, "expedia.com (Hilton Garden Inn Lima Miraflores)"),
        ("Fairfield by Marriott Lima Miraflores", 12, -12.125463, -77.021657, "amimagazine.global (Fairfield By Marriott Lima Miraflores)"),
    ]
    RASGOS_PISOS = ["altura_m", "altura_p90_m", "altura_ultimo_anio_m", "altura_p90_ultimo_anio_m", "altura_max_m",
                    "altura_vecindario_m", "log_area", "n_anios_altura"]
    MONOTONIA_PISOS = [1, 1, 1, 1, 1, 0, 0, 0]
    web = pd.DataFrame(REFERENCIA_WEB, columns=["nombre", "pisos_web", "lat", "lon", "fuente"])
    web_x, web_y = Transformer.from_crs(4326, EPSG_UTM_LIMA, always_xy=True).transform(web["lon"].to_numpy(), web["lat"].to_numpy())
    nombres_web = web["nombre"].tolist()


    # Busca, por lote, la huella que contiene a cada edificio de la lista web o está a menos de 25 m.
    def candidatos_web(lotes):
        puntos = shapely.points(web_x, web_y)
        for pos in lotes:
            arbol = shapely.STRtree(huellas_utm(pos["geometry_wkt"]))
            i_pt, i_h = arbol.query(puntos, predicate="within")
            (j_pt, j_h), dist_j = arbol.query_nearest(puntos, max_distance=25.0, return_distance=True)
            pt, hu = np.r_[i_pt, j_pt].astype(int), np.r_[i_h, j_h].astype(int)
            yield pd.DataFrame({"nombre": [nombres_web[k] for k in pt], "id": pos["id"].to_numpy()[hu],
                                "dentro": np.r_[np.ones(len(i_pt), bool), np.zeros(len(j_pt), bool)], "distancia": np.r_[np.zeros(len(i_pt)), dist_j]})


    # Columnas que usa el modelo de pisos.
    def tabla_rasgos(df):
        return df.select("id", F.log("area_m2").alias("log_area"), "altura_m", "altura_p90_m", "altura_ultimo_anio_m", "altura_p90_ultimo_anio_m",
                         "altura_max_m", "altura_vecindario_m", "n_anios_altura", "altura_metodo", "niveles_osm", "x_utm", "y_utm")


    elegidos = (huella.where(F.col("geometry_wkt").isNotNull()).mapInPandas(candidatos_web, "nombre string, id string, dentro boolean, distancia double").toPandas()
                .sort_values(["nombre", "dentro", "distancia", "id"], ascending=[True, False, True, True]).drop_duplicates("nombre"))
    rasgos = spark.table("silver_tmp_edificios_rasgos")
    web = web.merge(elegidos[["nombre", "id"]], on="nombre").merge(tabla_rasgos(rasgos.where(F.col("id").isin(elegidos["id"].tolist()))).toPandas(), on="id")
    web["huella_compartida"] = web.groupby("id")["nombre"].transform("size") > 1
    print(f"Referencia web: {len(REFERENCIA_WEB)} edificios | con huella: {len(web)} | en huella compartida: {web.huella_compartida.sum()}")

    osm_ref = (tabla_rasgos(rasgos.where((F.col("altura_metodo") == "huella_multianual") & F.col("niveles_osm").between(1, 45))).toPandas().sort_values("id"))
    web_ref = web.loc[~web.huella_compartida & (web.altura_metodo == "huella_multianual"), RASGOS_PISOS + ["pisos_web", "x_utm", "y_utm"]]
    ref_pisos = pd.concat([osm_ref[RASGOS_PISOS + ["niveles_osm", "x_utm", "y_utm"]].rename(columns={"niveles_osm": "pisos_ref"}).assign(fuente="osm", peso=1.0),
                           web_ref.rename(columns={"pisos_web": "pisos_ref"}).assign(fuente="web", peso=float(PESO_REFERENCIA_WEB))],
                          ignore_index=True)


    # Entrena el gradient boosting monótono y la recta altura-pisos.
    def ajustar_pisos(d):
        gbt = HistGradientBoostingRegressor(loss="absolute_error", max_iter=400, learning_rate=0.05, min_samples_leaf=15,
                                            monotonic_cst=MONOTONIA_PISOS, random_state=SEED).fit(d[RASGOS_PISOS], d.pisos_ref)
        costo = lambda p: np.sum(d.peso * np.abs((d.altura_m - p[0]) / p[1] - d.pisos_ref))
        a, b = minimize(costo, [0.0, ALTURA_POR_PISO_M], method="Nelder-Mead").x
        return gbt, float(a), float(b)


    # Convierte altura en pisos con la recta calibrada.
    def a_pisos(h, a, b):
        return np.maximum(1, np.round((np.asarray(h, float) - a) / b))


    # Pisos con gradient boosting hasta 40 m y con la recta para las torres.
    def predecir_pisos(modelo, d):
        gbt, a, b = modelo
        recta = d.altura_m.gt(ALTURA_CORTE_TORRE_M) | d.altura_p90_m.isna()
        return np.where(recta, a_pisos(d.altura_m, a, b), np.maximum(1, np.round(gbt.predict(d[RASGOS_PISOS]))))


    bloque = (ref_pisos.x_utm // 2000).astype(int).astype(str) + "_" + (ref_pisos.y_utm // 2000).astype(int).astype(str)
    ref_pisos["fold"] = bloque.map(lambda b: zlib.crc32(b.encode()) % 5)
    ref_pisos["pisos_antes"] = a_pisos(ref_pisos.altura_ultimo_anio_m, 0.0, ALTURA_POR_PISO_M)
    ref_pisos["pisos_calibrado"] = np.nan
    for f in range(5):
        en_f = ref_pisos.fold == f
        ref_pisos.loc[en_f, "pisos_calibrado"] = predecir_pisos(ajustar_pisos(ref_pisos[~en_f]), ref_pisos[en_f])
    modelo_pisos = ajustar_pisos(ref_pisos)
    _, A_PISO, B_PISO = modelo_pisos
    print(f"Hasta {ALTURA_CORTE_TORRE_M:.0f} m: gradient boosting monótono | sobre {ALTURA_CORTE_TORRE_M:.0f} m: pisos = (altura {-A_PISO:+.2f} m) / {B_PISO:.2f} m"
          f"  ({len(osm_ref):,} edificios de OSM + {len(web_ref)} de la web)")

    err_antes = ref_pisos.pisos_antes - ref_pisos.pisos_ref
    err_cal = ref_pisos.pisos_calibrado - ref_pisos.pisos_ref
    grupo = pd.cut(ref_pisos.pisos_ref, [0, 2, 4, 6, 9, 14, 99], labels=["1-2", "3-4", "5-6", "7-9", "10-14", "15+"])
    display(ref_pisos.assign(grupo=grupo, ea=err_antes.abs(), ec=err_cal.abs(), sa=err_antes, sc=err_cal,
                             ok_a=err_antes.abs() <= 1, ok_c=err_cal.abs() <= 1)
            .groupby(["fuente", "grupo"], observed=True)
            .agg(edificios=("ea", "size"), error_antes=("ea", "mean"), error_calibrado=("ec", "mean"), sesgo_antes=("sa", "mean"),
                 sesgo_calibrado=("sc", "mean"), pct_1piso_antes=("ok_a", "mean"), pct_1piso_calibrado=("ok_c", "mean"))
            .round(2).reset_index().astype({"fuente": str, "grupo": str}))
    for fuente, d in ref_pisos.groupby("fuente"):
        print(f"{fuente}: error medio {(d.pisos_antes - d.pisos_ref).abs().mean():.2f} → {(d.pisos_calibrado - d.pisos_ref).abs().mean():.2f} pisos")
    web_ok = web[web.altura_m.notna()]
    display(web_ok.assign(pisos_antes=a_pisos(web_ok.altura_ultimo_anio_m, 0.0, ALTURA_POR_PISO_M), pisos_calibrado=predecir_pisos(modelo_pisos, web_ok))
            [["nombre", "pisos_web", "pisos_antes", "pisos_calibrado", "altura_ultimo_anio_m", "altura_m", "huella_compartida", "fuente"]]
            .round(1))


    # Aplica el modelo de pisos a cada lote de edificios.
    def pisos_lote(lotes):
        for d in lotes:
            d["pisos_medidos"] = np.nan
            con_altura = d["altura_m"].notna().to_numpy()
            if con_altura.any():
                d.loc[con_altura, "pisos_medidos"] = predecir_pisos(modelo_pisos, d[con_altura])
            yield d[["id", "pisos_medidos"]]


    pisos_df = tabla_rasgos(rasgos).mapInPandas(pisos_lote, "id string, pisos_medidos double")
    escribir(rasgos.join(pisos_df, "id", "left").select("id", "x_utm", "y_utm", "tesela", "altura_m", "altura_max_m", "altura_ultimo_anio_m", "n_anios_altura",
                                                        "altura_metodo", "niveles_osm", "pisos_medidos", "vs30", "zona_cismid"), "silver_tmp_edificios_altura")
    (spark.table("silver_tmp_edificios_altura").withColumn("pisos_10", F.least("pisos_medidos", F.lit(10.0))).groupBy("pisos_10").count().orderBy("pisos_10").show())

In [ ]:
# Une altura, suelo y pisos; calcula el periodo Tb y guarda silver_edificios.
if procesar_maestros:
    CT_ALBANILERIA, CT_CONCRETO, PISOS_CONCRETO_TB = 60.0, 45.0, 5
    e030_suelo = F.broadcast(spark.table("silver_e030_parametros").select(F.col("perfil").alias("tipo_suelo"), "S",
                                                                     F.col("TP_s").alias("TP"), F.col("TL_s").alias("TL")))
    PERFIL_ZONA = {1: "S1", 2: "S2", 3: "S3", 4: "S3", 5: "S3"}
    perfil_zona = F.lit(None).cast("string")
    for zona_c, perfil_c in PERFIL_ZONA.items():
        perfil_zona = F.when(F.col("zona_cismid") == zona_c, F.lit(perfil_c)).otherwise(perfil_zona)
    con_altura = (edif.join(spark.table("silver_tmp_edificios_altura"), "id", "left")
                  .withColumn("tipo_suelo", F.coalesce(perfil_zona, perfil_e030(F.col("vs30"))))
                  .withColumn("perfil_fuente", F.when(perfil_zona.isNotNull(), "microzonificacion_cismid").otherwise("vs30_usgs"))
                  .join(e030_suelo, "tipo_suelo", "left"))
    con_altura = con_altura.withColumn("pisos_medidos", F.col("pisos_medidos").cast("int"))
    mediana_r9 = con_altura.groupBy("h3_id").agg(F.expr("percentile_approx(pisos_medidos, 0.5)").alias("pisos_h3_r9"))
    mediana_r8 = con_altura.groupBy("h3_r8").agg(F.expr("percentile_approx(pisos_medidos, 0.5)").alias("pisos_h3_r8"))
    osm_muy_alto = F.coalesce((F.col("niveles_osm") >= OSM_MIN_PISOS_CONTRASTE)
                              & (F.col("altura_max_m") < 0.5 * ALTURA_POR_PISO_M * F.least(F.col("niveles_osm"), F.lit(float(OSM_PISOS_SATURACION)))),
                              F.lit(False))
    osm_muy_bajo = F.coalesce(F.col("altura_m") > 3 * ALTURA_POR_PISO_M * F.col("niveles_osm") + ALTURA_POR_PISO_M, F.lit(False))
    edificios = (con_altura.join(mediana_r9, "h3_id", "left").join(mediana_r8, "h3_r8", "left")
        .withColumn("altura_imputada", F.col("altura_m").isNull())
        .withColumn("pisos_25d", F.coalesce("pisos_medidos", "pisos_h3_r9", "pisos_h3_r8", F.lit(1)).cast("int"))
        .withColumn("osm_aceptado", F.col("niveles_osm").isNotNull() & ~osm_muy_alto & ~osm_muy_bajo)
        .withColumn("pisos", F.when(F.col("osm_aceptado"), F.greatest(F.lit(1.0), F.round("niveles_osm")))
                              .otherwise(F.col("pisos_25d")).cast("int"))
        .withColumn("pisos_metodo", F.when(F.col("osm_aceptado"), "osm")
                                     .when(~F.col("altura_imputada"), "medido")
                                     .when(F.col("pisos_h3_r9").isNotNull(), "mediana_h3_r9")
                                     .when(F.col("pisos_h3_r8").isNotNull(), "mediana_h3_r8").otherwise("1_por_defecto"))
        .withColumn("altura_tb_m", F.when(F.col("osm_aceptado"), F.greatest(F.coalesce("altura_m", F.lit(0.0)), F.col("pisos") * ALTURA_POR_PISO_M))
                                    .when(F.col("altura_imputada"), F.col("pisos") * ALTURA_POR_PISO_M)
                                    .otherwise(F.greatest(F.col("altura_m"), F.lit(ALTURA_POR_PISO_M))))
        .withColumn("Tb", F.round(F.col("altura_tb_m") / F.when(F.col("pisos") >= PISOS_CONCRETO_TB, CT_CONCRETO)
                                                         .otherwise(CT_ALBANILERIA), 3))
        .withColumn("Tb_TP", F.round(F.col("Tb") / F.col("TP"), 3))
        .withColumn("con_microzonificacion", F.col("distrito").isin(list(DISTRITO_PDF.values())))
        .select("id", "ubigeo", "distrito", "provincia", "h3_id", "h3_r8", "lon", "lat", "x_utm", "y_utm",
                "area_m2", "confianza", "altura_m", "altura_max_m", "altura_metodo", "altura_imputada", "pisos", "pisos_metodo",
                "pisos_25d", "niveles_osm", "altura_ultimo_anio_m", "n_anios_altura", "Tb",
                "vs30", "zona_cismid", "tipo_suelo", "perfil_fuente", "S", "TP", "TL", "Tb_TP", "con_microzonificacion", "tesela",
                "geometry_wkt"))
    n_edif = escribir(edificios, "silver_edificios", particion="provincia")

In [ ]:
# Registro de calidad, cuarentena y limpieza de temporales.
if procesar_maestros:
    e = spark.table("silver_edificios")
    agregados = problemas([
        {"fuente": "open_buildings_v3", "regla": "fuera_del_rectangulo_estudio", "accion": "fuera_de_alcance", "registro_id": None,
         "detalle": "edificio de la tesela 911 fuera del rectángulo de los distritos de estudio", "n_filas": n_bronze - n_bbox},
        {"fuente": "open_buildings_v3", "regla": "fuera_de_distritos_estudio", "accion": "fuera_de_alcance", "registro_id": None,
         "detalle": "centroide dentro del rectángulo pero fuera de los 6 distritos de estudio (ST_Within)", "n_filas": n_bbox - n_lima},
        {"fuente": "open_buildings_v3", "regla": "id_duplicado", "accion": "descartado", "registro_id": None,
         "detalle": "full_plus_code repetido; se conserva una fila", "n_filas": n_dups},
        {"fuente": "osm_niveles", "regla": "pisos_osm", "accion": "reemplazado", "registro_id": None,
         "detalle": "pisos tomados de building:levels de OSM (punto dentro de la huella)", "n_filas": e.where("pisos_metodo = 'osm'").count()},
        {"fuente": "osm_niveles", "regla": "osm_contradice_2.5D", "accion": "descartado", "registro_id": None,
         "detalle": "pisos de OSM incompatibles con la altura 2.5D; se usa la estimación 2.5D",
         "n_filas": e.where("niveles_osm IS NOT NULL AND pisos_metodo <> 'osm'").count()},
    ])
    por_fila = (baja_conf.select(F.lit("open_buildings_v3").alias("fuente"), F.lit(f"confianza<{CONFIANZA_MIN}").alias("regla"),
                                 F.lit("descartado").alias("accion"), F.col("id").alias("registro_id"),
                                 F.format_string("confianza=%.3f", "confianza").alias("detalle"), F.lit(1).alias("n_filas"))
                .unionByName(e.where("altura_imputada").select(
                    F.lit("open_buildings_25d").alias("fuente"), F.lit("sin_altura_2.5D").alias("regla"), F.lit("imputado").alias("accion"),
                    F.col("id").alias("registro_id"), F.concat(F.lit("pisos="), F.col("pisos").cast("string"), F.lit(" ("), F.col("pisos_metodo"), F.lit("); tesela="),
                             F.coalesce("tesela", F.lit("ninguna"))).alias("detalle"),
                    F.lit(1).alias("n_filas")))
                .unionByName(e.where("vs30 IS NULL").select(
                    F.lit("vs30_usgs").alias("fuente"), F.lit("edificio_sin_vs30").alias("regla"), F.lit("advertencia").alias("accion"),
                    F.col("id").alias("registro_id"), F.lit("el píxel de Vs30 no existe en la ventana").alias("detalle"), F.lit(1).alias("n_filas"))))
    registrar_calidad(agregados.unionByName(por_fila), "edificios")

    escribir(baja_conf.select("id", "lon", "lat", "area_m2", "confianza", "ubigeo", "distrito",
                              F.lit(f"confianza<{CONFIANZA_MIN}").alias("motivo_rechazo")), "silver_edificios_cuarentena")

    for tmp in ["silver_tmp_edificios_distrito", "silver_tmp_edificios_altura", "silver_tmp_edificios_huella", "silver_tmp_alturas_tesela", "silver_tmp_edificios_rasgos"]:
        spark.sql(f"DROP TABLE IF EXISTS {tmp}")
    print("Tablas temporales eliminadas")

In [ ]:
# Contrasta los pisos estimados con los de OSM.
if procesar_maestros:
    e = spark.table("silver_edificios")
    par = (e.where("niveles_osm IS NOT NULL")
           .select("id", "lon", "lat", "distrito", F.col("niveles_osm").alias("niveles"), "pisos_25d", "pisos", "pisos_metodo",
                   "altura_metodo", "altura_m", "altura_max_m", (F.col("pisos_metodo") == "osm").alias("osm_aceptado")))
    escribir(par, "silver_validacion_pisos_osm")
    par = spark.table("silver_validacion_pisos_osm")
    ok = par.where("osm_aceptado").withColumn("dif", F.col("pisos_25d") - F.col("niveles"))
    r = ok.agg(F.count("*").alias("n"), F.avg((F.col("dif") == 0).cast("double")).alias("exactos"), F.avg((F.abs("dif") <= 1).cast("double")).alias("uno"),
               F.avg(F.abs("dif")).alias("error"), F.avg("dif").alias("sesgo")).first()
    print(f"Edificios con pisos de OSM: {par.count():,} | aceptados: {r.n:,} | descartados (contradicen al 2.5D): {par.count() - r.n:,}")
    print(f"Estimación 2.5D vs OSM: pisos exactos {100 * r.exactos:.1f} % | ±1 piso {100 * r.uno:.1f} % | error medio {r.error:.2f} pisos | sesgo {r.sesgo:+.2f}")
    grupo = (F.when(F.col("niveles") <= 1, "1").when(F.col("niveles") <= 2, "2").when(F.col("niveles") <= 3, "3").when(F.col("niveles") <= 4, "4")
             .when(F.col("niveles") <= 6, "5-6").when(F.col("niveles") <= 9, "7-9").when(F.col("niveles") <= 14, "10-14").otherwise("15+"))
    display(ok.withColumn("grupo", grupo).groupBy("grupo")
            .agg(F.count("*").alias("edificios"), F.round(F.avg("niveles"), 2).alias("pisos_osm"), F.round(F.avg("pisos_25d"), 2).alias("pisos_25d"),
                 F.round(F.avg("altura_m"), 2).alias("altura_2_5d_m"), F.round(F.avg(F.abs("dif")), 2).alias("error_medio"), F.round(F.avg("dif"), 2).alias("sesgo"))
            .orderBy("pisos_osm"))
    tabla_cruce = (ok.withColumn("pisos_osm", F.least("niveles", F.lit(6.0))).withColumn("pisos_2_5d", F.least("pisos_25d", F.lit(6)))
                   .groupBy("pisos_osm", "pisos_2_5d").count()
                   .withColumn("fraccion", F.round(F.col("count") / F.sum("count").over(Window.partitionBy("pisos_osm")), 2)))
    display(tabla_cruce.groupBy("pisos_osm").pivot("pisos_2_5d").agg(F.first("fraccion")).fillna(0.0).orderBy("pisos_osm"))

### STEAD

In [ ]:
if procesar_maestros:
    FS_STEAD = 100.0
    sd = spark.table("bronze_stead_metadata")
    snr = F.split(F.regexp_replace("snr_db", r"[\[\]]", ""), r"\s+")


    # Relación señal-ruido de un canal.
    def snr_db(k):
        return F.nanvl(F.col("snr_arr")[k].cast("double"), F.lit(None).cast("double"))


    stead = (sd
        .withColumn("snr_arr", F.filter(snr, lambda s: s != ""))
        .select("trace_name", "trace_category", (F.col("trace_category") == "earthquake_local").alias("es_sismo"),
                "network_code", "receiver_code", "receiver_type", "receiver_latitude", "receiver_longitude",
                "p_arrival_sample", "p_status", "p_weight", "s_arrival_sample", "s_status", "s_weight",
                (F.col("p_arrival_sample") / FS_STEAD).alias("p_segundos"), (F.col("s_arrival_sample") / FS_STEAD).alias("s_segundos"),
                ((F.col("s_arrival_sample") - F.col("p_arrival_sample")) / FS_STEAD).alias("s_menos_p_s"),
                "source_magnitude", "source_magnitude_type", "source_distance_km", "source_depth_km",
                "source_latitude", "source_longitude", "back_azimuth_deg",
                snr_db(0).alias("snr_e_db"), snr_db(1).alias("snr_n_db"), snr_db(2).alias("snr_z_db"),
                "coda_end_sample", "trace_start_time", "_archivo_hdf5"))
    fallas = stead.where(F.col("es_sismo") & (F.col("p_arrival_sample").isNull() | F.col("source_magnitude").isNull()))
    stead_ok = stead.join(fallas.select("trace_name"), "trace_name", "left_anti")
    escribir(stead_ok, "silver_stead_trazas")
    registrar_calidad(fallas.select(
        F.lit("stead").alias("fuente"), F.lit("sismo_sin_llegada_P_o_magnitud").alias("regla"), F.lit("descartado").alias("accion"),
        F.col("trace_name").alias("registro_id"), F.lit("no sirve para entrenar el detector de onda P").alias("detalle"),
        F.lit(1).alias("n_filas")), "stead_trazas")

    escribir(fallas.select("trace_name", "trace_category", "p_arrival_sample", "source_magnitude",
                           F.lit("sismo_sin_llegada_P_o_magnitud").alias("motivo_rechazo")), "silver_stead_cuarentena")

    s = spark.table("silver_stead_trazas")
    s.groupBy("trace_category").agg(F.count("*").alias("trazas"), F.round(F.avg("source_magnitude"), 2).alias("mag_media"),
                                    F.round(F.expr("percentile_approx(p_segundos, 0.5)"), 1).alias("p_mediana_s"),
                                    F.round(F.avg("snr_z_db"), 1).alias("snr_z_medio")).show()

In [ ]:
if procesar_maestros:
    N_TRAZAS = 10_000
    FS_STEAD = 100
    ANTES, DESPUES = 100, 300
    archivo_hdf5 = (MAESTROS / "stead" / "chunk2_muestra.hdf5").as_posix()
    RASGOS_VENTANA = [f"{r}_{c}" for c in "enz" for r in ("log_energia", "log_max", "curtosis", "log_ratio_mitades")] + ["frec_dominante_z", "log_z_sobre_h"]
    ESQ_VENTANAS = "trace_name string, tipo string, etiqueta long, " + ", ".join(f"{c} double" for c in RASGOS_VENTANA)
    ESQ_PICKS = "trace_name string, p_s double, pick_sta_lta_s double, magnitud double, distancia_km double"


    # Energía, máximo, curtosis y frecuencia dominante de una ventana de 4 s.
    def rasgos_ventana(x):
        x = x - x.mean(0)
        mitad = len(x) // 2
        e1 = (x[:mitad] ** 2).mean(0) + 1e-12
        e2 = (x[mitad:] ** 2).mean(0) + 1e-12
        r = {}
        for i, c in enumerate("enz"):
            r[f"log_energia_{c}"] = float(np.log10((x[:, i] ** 2).mean() + 1e-12))
            r[f"log_max_{c}"] = float(np.log10(np.abs(x[:, i]).max() + 1e-12))
            r[f"curtosis_{c}"] = float(kurtosis(x[:, i]))
            r[f"log_ratio_mitades_{c}"] = float(np.log10(e2[i] / e1[i]))
        espectro = np.abs(np.fft.rfft(x[:, 2]))
        r["frec_dominante_z"] = float(np.fft.rfftfreq(len(x), 1 / FS_STEAD)[espectro.argmax()])
        r["log_z_sobre_h"] = float(np.log10(((x[:, 2] ** 2).mean() + 1e-12) / ((x[:, :2] ** 2).mean() + 1e-12)))
        return r


    # Detecta la llegada de la onda P con STA/LTA clásico.
    def picar_sta_lta(z, sta=50, lta=500, umbral=4.0):
        z = z - z.mean()
        e = np.concatenate([[0.0], np.cumsum(z * z)])
        i = np.arange(lta, len(z) - sta)
        cociente = ((e[i + sta] - e[i]) / sta) / ((e[i] - e[i - lta]) / lta + 1e-20)
        sobre = np.nonzero(cociente > umbral)[0]
        return (i[sobre[0]] + sta) / FS_STEAD if sobre.size else np.nan


    # Se ejecuta por partición: rasgos de las ventanas de P, ruido, coda y S de cada traza.
    def ventanas_traza(lotes):
        with h5py.File(archivo_hdf5, "r") as h:
            for pdf in lotes:
                filas = []
                for r in pdf.itertuples():
                    x = np.asarray(h["data"][r.trace_name], dtype="float64")
                    p, s = int(r.p_arrival_sample), int(r.s_arrival_sample)
                    rng_v = np.random.default_rng(zlib.crc32(r.trace_name.encode()))
                    filas.append({"trace_name": r.trace_name, "tipo": "onda_P", "etiqueta": 1, **rasgos_ventana(x[p - ANTES: p + DESPUES])})
                    filas.append({"trace_name": r.trace_name, "tipo": "ruido_previo", "etiqueta": 0, **rasgos_ventana(x[p - ANTES - 400: p - ANTES])})
                    i0 = p + DESPUES + int(rng_v.integers(0, max(1, min(6000 - 400, p + 1500) - (p + DESPUES))))
                    if i0 + 400 <= 6000:
                        filas.append({"trace_name": r.trace_name, "tipo": "dentro_del_sismo", "etiqueta": 0, **rasgos_ventana(x[i0: i0 + 400])})
                    if s - ANTES >= p + DESPUES and s + DESPUES <= 6000:
                        filas.append({"trace_name": r.trace_name, "tipo": "llegada_S", "etiqueta": 0, **rasgos_ventana(x[s - ANTES: s + DESPUES])})
                yield pd.DataFrame(filas, columns=["trace_name", "tipo", "etiqueta"] + RASGOS_VENTANA)


    # Se ejecuta por partición: pick STA/LTA de cada traza del HDF5.
    def picks_traza(lotes):
        with h5py.File(archivo_hdf5, "r") as h:
            for pdf in lotes:
                yield pd.DataFrame([{"trace_name": r.trace_name, "p_s": int(r.p_arrival_sample) / FS_STEAD,
                                     "pick_sta_lta_s": picar_sta_lta(np.asarray(h["data"][r.trace_name], dtype="float64")[:, 2]),
                                     "magnitud": r.source_magnitude, "distancia_km": r.source_distance_km} for r in pdf.itertuples()],
                                   columns=["trace_name", "p_s", "pick_sta_lta_s", "magnitud", "distancia_km"])


    muestra = (spark.table("silver_stead_trazas")
               .where(f"p_arrival_sample >= {ANTES + 500} AND p_arrival_sample <= {6000 - DESPUES} AND s_arrival_sample IS NOT NULL")
               .select("trace_name", "p_arrival_sample", "s_arrival_sample", "source_magnitude", "source_distance_km")
               .orderBy(F.rand(1)).limit(N_TRAZAS))
    escribir(muestra, "silver_tmp_stead_muestra")
    muestra = spark.table("silver_tmp_stead_muestra").repartition(32)
    escribir(muestra.mapInPandas(ventanas_traza, ESQ_VENTANAS), "silver_tmp_stead_ventanas")
    escribir(muestra.mapInPandas(picks_traza, ESQ_PICKS).withColumn("error_s", F.col("pick_sta_lta_s") - F.col("p_s")), "silver_tmp_stead_picks")

    ventanas = spark.table("silver_tmp_stead_ventanas")
    sin_definir = F.lit(False)
    for c in RASGOS_VENTANA:
        sin_definir = sin_definir | F.col(c).isNull() | F.isnan(c) | F.col(c).isin(float("inf"), float("-inf"))
    canal_sin_senal = sorted(r.trace_name for r in ventanas.where(sin_definir).select("trace_name").distinct().collect())
    registrar_calidad(spark.createDataFrame(
        [("stead", "canal_sin_senal", "cuarentena", t, "rasgos de la ventana sin definir (curtosis nula)", 1) for t in canal_sin_senal],
        ESQUEMA_PROBLEMAS), "stead_ventanas")
    if canal_sin_senal:
        escribir(spark.table("silver_stead_trazas").where(F.col("trace_name").isin(canal_sin_senal))
                 .select("trace_name", "trace_category", "p_arrival_sample", "source_magnitude",
                         F.lit("canal_sin_senal").alias("motivo_rechazo")), "silver_stead_cuarentena", modo="append")
    print("Trazas a cuarentena por canal sin señal:", canal_sin_senal)

    sano = ~F.col("trace_name").isin(canal_sin_senal)
    escribir(ventanas.where(sano), "silver_stead_ventanas")
    escribir(spark.table("silver_tmp_stead_picks").where(sano), "silver_stead_picks")
    spark.table("silver_stead_ventanas").groupBy("tipo").count().show()
    for tmp in ["silver_tmp_stead_muestra", "silver_tmp_stead_ventanas", "silver_tmp_stead_picks"]:
        spark.sql(f"DROP TABLE IF EXISTS {tmp}")

### Edificios y ondas de Tainan y Manta

In [ ]:
if procesar_maestros:
    RADIO_TAINAN_KM = 40.0
    px_ciudades = spark.table("bronze_vs30_pixeles_ciudades")


    # Convierte a número; si no se puede, deja nulo.
    def numero(columna):
        return F.col(columna).try_cast("double")


    # Vs30 del píxel más cercano en Tainan o Manta, con join por índice de malla.
    def con_vs30_ciudad(df, ciudad):
        g = px_ciudades.where(F.col("ciudad") == ciudad)
        m = g.agg(F.min("fila").alias("f0"), F.max("fila").alias("f1"), F.min("col").alias("c0"), F.max("col").alias("c1"),
                  F.max("lat").alias("lat0"), F.min("lat").alias("lat1"), F.min("lon").alias("lon0"), F.max("lon").alias("lon1")).first()
        i = F.floor((m.lat0 - F.col("lat")) / ((m.lat0 - m.lat1) / (m.f1 - m.f0)) + 0.5)
        j = F.floor((F.col("lon") - m.lon0) / ((m.lon1 - m.lon0) / (m.c1 - m.c0)) + 0.5)
        pixel = g.select(F.col("fila").alias("_f"), F.col("col").alias("_c"), F.col("vs30").alias("_vs"))
        return (df.withColumn("_f", F.least(F.greatest(i, F.lit(0)), F.lit(m.f1 - m.f0)).cast("long") + m.f0)
                .withColumn("_c", F.least(F.greatest(j, F.lit(0)), F.lit(m.c1 - m.c0)).cast("long") + m.c0)
                .join(F.broadcast(pixel), ["_f", "_c"], "left")
                .withColumn("vs30", F.when((F.col("_vs") > 0) & (F.col("_vs") < 3000), F.col("_vs")))
                .drop("_f", "_c", "_vs"))


    # Agrega altura, periodo Tb, etiqueta de daño severo y código de suelo.
    def completar_edificio(df):
        return (df.withColumn("pisos", F.col("pisos").cast("long"))
                .withColumn("altura_m", F.col("pisos") * 3.0)
                .withColumn("Tb", F.bround(F.greatest(F.lit(0.05), F.col("altura_m") / F.when(F.col("pisos") >= 5, 45.0).otherwise(60.0)), 3))
                .withColumn("severo", F.coalesce(F.col("dano").isin("S", "C"), F.lit(False)))
                .withColumn("suelo_cod", F.when(F.col("vs30") < 180, 3.0).when(F.col("vs30") < 500, 2.0).when(F.col("vs30") < 1500, 1.0).otherwise(0.0)))


    casos = (spark.table("bronze_tainan_casos").select(numero("datahub_id").alias("datahub_id"), F.col("Longitude").alias("lon_caso"))
             .where("datahub_id IS NOT NULL").groupBy("datahub_id").agg(F.first("lon_caso", ignorenulls=True).alias("lon_caso")))
    lon_t, pisos_t = numero("Longitude"), numero("No_Floors")
    tainan = (spark.table("bronze_tainan_edificios").withColumn("datahub_id", numero("datahub_id")).join(casos, "datahub_id", "left")
              .select(F.lit("Tainan").alias("ciudad"), F.col("Experiment_ID").alias("id"), numero("Latitude").alias("lat"),
                      F.when(lon_t >= 100, lon_t).otherwise(numero("lon_caso")).alias("lon"), F.col("Structural_Damage").alias("dano"), pisos_t.alias("pisos"),
                      F.coalesce(numero("First_Floor_Area"), F.try_divide(numero("Total_Floor_Area"), pisos_t)).alias("area_m2"))
              .dropna(subset=["lat", "lon", "dano", "pisos", "area_m2"])
              .where(F.col("dano").isin(list("NLMSC")))
              .withColumn("pisos", F.greatest(F.floor(F.col("pisos") + 0.5), F.lit(1)).cast("double")))
    tainan = completar_edificio(con_vs30_ciudad(tainan, "Tainan").dropna(subset=["vs30"]))

    manta = (spark.table("bronze_ecuador_edificios")
             .select(F.lit("Manta").alias("ciudad"), F.col("Experiment_ID").alias("id"), numero("Latitude").alias("lat"), numero("Longitude").alias("lon"),
                     F.col("Structural_Damage").alias("dano"), numero("No_Floors").alias("pisos"), numero("Floor_Area").alias("area_m2"))
             .where(F.col("lon").between(-80.80, -80.62) & F.col("lat").between(-1.03, -0.90))
             .dropna(subset=["pisos", "area_m2"]))
    manta = completar_edificio(con_vs30_ciudad(manta, "Manta"))

    COLS_DANO_EDIF = ["ciudad", "id", "lat", "lon", "dano", "severo", "pisos", "altura_m", "area_m2", "Tb", "vs30", "suelo_cod"]
    escribir(tainan.select(*COLS_DANO_EDIF).unionByName(manta.select(*COLS_DANO_EDIF)), "silver_dano_edificios")
    display(spark.table("silver_dano_edificios").groupBy("ciudad")
            .agg(F.count("*").alias("edificios"), F.round(F.avg(F.col("severo").cast("double")), 2).alias("severos"), F.median("pisos").alias("pisos"),
                 F.round(F.median("area_m2"), 2).alias("area"), F.round(F.median("vs30"), 2).alias("vs30")))

In [ ]:
if procesar_maestros:
    ESQ_ONDAS = ("ciudad string, evento string, archivo_estacion string, estacion string, lat double, lon double, vs30 double, fs double, pga_geo_g double, "
                 "pgv_geo_cms double, tp_s double, sa_max_geo_g double, sa_geo array<double>, arias_ms double, d595_s double")
    COLS_ONDAS = [c.split()[0] for c in ESQ_ONDAS.split(", ")]
    LON_AMNT, LAT_AMNT = -80.735, -0.941


    # Pasa lon y lat a UTM 51N.
    @F.pandas_udf("x double, y double")
    def utm_tainan(lon: pd.Series, lat: pd.Series) -> pd.DataFrame:
        x, y = Transformer.from_crs(4326, 32651, always_xy=True).transform(lon.to_numpy(float), lat.to_numpy(float))
        return pd.DataFrame({"x": x, "y": y}, index=lon.index)


    # Variables de onda de cada registro de Tainan.
    def ondas_tainan(lotes):
        for pdf in lotes:
            filas = []
            for f in pdf.itertuples():
                fs = float(re.search(r"SampleRate\(Hz\):\s*([\d.]+)", f.contenido[:800]).group(1))
                a = pd.read_csv(io.StringIO(f.contenido), sep=r"\s+", comment="#", header=None, engine="python").to_numpy(float)
                if a.shape[1] == 4 and len(a) >= 20 * fs:
                    filas.append({"ciudad": "Tainan", "evento": "2016-02-06_meinong_Mw6.4", "archivo_estacion": f.archivo, "estacion": f.estacion, "lat": f.lat, "lon": f.lon,
                                  "vs30": f.vs30, "fs": fs, **variables_onda(a[:, 2], a[:, 3], fs)})
            yield pd.DataFrame(filas, columns=COLS_ONDAS)


    # Frecuencia de muestreo y datos de un registro de RENAC.
    def leer_renac(texto):
        fs = float(re.search(r"muestreo \(Hz\):\s*([0-9.eE+-]+)", texto).group(1))
        return fs, np.array(texto.split("____")[-1].split("\n", 1)[1].split(), float)


    # Variables de onda de la estación de Manta.
    def onda_manta(lotes):
        for pdf in lotes:
            filas = []
            for r in pdf.itertuples():
                fs_m, e_m = leer_renac(r.txt_e)
                _, n_m = leer_renac(r.txt_n)
                filas.append({"ciudad": "Manta", "evento": "2016-04-16_ecuador_Mw7.8", "archivo_estacion": "AMNT", "estacion": "AMNT", "lat": r.lat, "lon": r.lon,
                              "vs30": r.vs30, "fs": fs_m, **variables_onda(e_m, n_m, fs_m)})
            yield pd.DataFrame(filas, columns=COLS_ONDAS)


    lista = (spark.table("bronze_tainan_lista_registros").withColumn("fila", numero("fila")).where("fila > 1")
             .withColumn("_r", F.row_number().over(Window.partitionBy("A").orderBy("libro", "fila"))).where("_r = 1"))
    archivo_lista = F.regexp_extract("K", r"([0-9A-Za-z]+(?:\.[0-9A-Za-z]+)*\.txt)", 1)
    estaciones = (lista.select(F.trim("A").alias("estacion"), F.regexp_extract("J", r"([\d.]+)\D+([\d.]+)", 1).try_cast("double").alias("lat"),
                               F.regexp_extract("J", r"([\d.]+)\D+([\d.]+)", 2).try_cast("double").alias("lon"), F.when(archivo_lista != "", archivo_lista).alias("archivo"))
                  .dropna(subset=["lat", "lon", "archivo"]))

    edif_tainan = (spark.table("silver_dano_edificios").where(F.col("ciudad") == "Tainan").withColumn("_b", utm_tainan("lon", "lat"))
                   .select(F.col("_b.x").alias("bx"), F.col("_b.y").alias("by")))
    cercanas = (estaciones.select("estacion", "archivo", "lat", "lon").withColumn("_e", utm_tainan("lon", "lat")).crossJoin(F.broadcast(edif_tainan))
                .groupBy("estacion", "archivo", "lat", "lon")
                .agg(F.min(F.sqrt((F.col("_e.x") - F.col("bx")) ** 2 + (F.col("_e.y") - F.col("by")) ** 2) / 1000).alias("dist_km"))
                .where(F.col("dist_km") <= RADIO_TAINAN_KM).drop("dist_km"))
    ondas_t = (con_vs30_ciudad(cercanas, "Tainan").where("vs30 IS NOT NULL")
               .join(spark.table("bronze_tainan_registros").select("archivo", "contenido"), "archivo")
               .repartition(16).mapInPandas(ondas_tainan, ESQ_ONDAS))

    amnt = spark.table("bronze_manta_registros").where(F.col("archivo").startswith("AMNT"))


    # Texto de una componente de la estación de Manta.
    def componente_amnt(marca, nombre):
        return amnt.where(F.col("archivo").contains(marca)).orderBy("archivo").limit(1).select(F.col("contenido").alias(nombre))


    punto_amnt = con_vs30_ciudad(spark.createDataFrame([(LON_AMNT, LAT_AMNT)], "lon double, lat double"), "Manta")
    ondas_m = (componente_amnt("_E_", "txt_e").crossJoin(componente_amnt("_N_", "txt_n")).crossJoin(punto_amnt).mapInPandas(onda_manta, ESQ_ONDAS))
    escribir(ondas_t.unionByName(ondas_m), "silver_dano_registros")

### Suelo no lineal y fragilidad

In [ ]:
# Coeficientes BSSA14 como números.
if procesar_maestros:
    escribir(spark.table("bronze_bssa14_no_lineal").select(*[F.col(c).cast("double") for c in ("T_s", "f4", "f5")]), "silver_bssa14_no_lineal")

In [ ]:
if procesar_maestros:
    ESTADOS = ["leve", "moderado", "severo", "colapso"]
    FRAGILIDAD_GEM = {
        "adobe": ("MUR-ADO/LWAL-DNO", [
            ( 1, "SA(0.3s)", 0.249, 0.621, 0.918, 1.168, 0.584),
            ( 2, "SA(0.3s)", 0.253, 0.642, 0.987, 1.291, 0.616),
            ( 3, "SA(0.6s)", 0.109, 0.302, 0.487, 0.658, 0.496),
        ]),
        "quincha": ("W-WWD/LWAL-DNO", [
            ( 1, "SA(0.3s)", 0.280, 0.679, 0.985, 1.237, 0.553),
            ( 2, "SA(0.3s)", 0.255, 0.722, 1.118, 1.463, 0.660),
        ]),
        "madera": ("W/LFM-DUL", [
            ( 1, "SA(0.3s)", 0.620, 1.128, 1.637, 2.112, 0.430),
            ( 2, "SA(0.3s)", 0.691, 1.346, 2.001, 2.616, 0.510),
        ]),
        "albanileria_confinada": ("MCF/LWAL-DUL", [
            ( 1, "PGA", 0.565, 0.941, 1.200, 1.400, 0.370),
            ( 2, "SA(0.3s)", 0.427, 1.205, 1.906, 2.535, 0.648),
            ( 3, "SA(0.3s)", 0.458, 1.109, 1.679, 2.181, 0.597),
            ( 4, "SA(0.6s)", 0.223, 0.545, 0.856, 1.144, 0.530),
            ( 5, "SA(0.6s)", 0.261, 0.578, 0.890, 1.181, 0.537),
            ( 6, "SA(0.6s)", 0.327, 0.639, 0.947, 1.236, 0.539),
        ]),
        "concreto_armado": ("CR/LFINF-DUM", [
            ( 5, "SA(0.6s)", 0.305, 0.946, 1.566, 2.144, 0.553),
            ( 6, "SA(0.6s)", 0.380, 1.053, 1.711, 2.326, 0.555),
            ( 7, "SA(1.0s)", 0.227, 0.577, 0.929, 1.262, 0.553),
            ( 8, "SA(1.0s)", 0.290, 0.650, 1.013, 1.357, 0.540),
            ( 9, "SA(1.0s)", 0.363, 0.719, 1.077, 1.413, 0.514),
            (10, "SA(1.0s)", 0.443, 0.790, 1.138, 1.464, 0.501),
        ]),
    }
    FUENTE_FRAGILIDAD = ("Martins L, Silva V (2021). Development of a fragility and vulnerability model for global seismic risk "
                         "analyses. Bulletin of Earthquake Engineering. doi:10.1007/s10518-020-00885-1 | "
                         "github.com/lmartins88/global_fragility_vulnerability (CC BY-SA 4.0)")
    COLS_THETA = [f"theta_{e}" for e in ESTADOS]
    ESQ_GEM = "archivo string, im_gem string, " + ", ".join(f"{c}_gem double" for c in COLS_THETA) + ", beta_gem double"


    # Ajusta la curva lognormal de un archivo GEM: mediana por estado y beta promedio.
    def ajustar_gem(g):
        d = pd.read_csv(io.StringIO(g["contenido"].iloc[0]))
        x = d.iloc[:, 0].to_numpy()
        thetas, betas = [], []
        for col in d.columns[1:]:
            p = d[col].to_numpy()
            m = (p > 0.02) & (p < 0.98)
            pendiente, corte = np.polyfit(np.log(x[m]), norm.ppf(p[m]), 1)
            thetas.append(round(float(np.exp(-corte / pendiente)), 3))
            betas.append(1 / pendiente)
        return pd.DataFrame([{"archivo": g["archivo"].iloc[0], "im_gem": d.columns[0].split(" ")[0],
                              **{f"{c}_gem": t for c, t in zip(COLS_THETA, thetas)}, "beta_gem": round(float(np.mean(betas)), 3)}])


    base_frag = spark.createDataFrame(
        [(t, tax, p, im, *th, b, f"{tax.replace('/', '_')}_H{p}.csv") for t, (tax, filas) in FRAGILIDAD_GEM.items() for p, im, *th, b in filas],
        "tipologia string, taxonomia_gem string, pisos_gem long, im string, " + ", ".join(f"{c} double" for c in COLS_THETA) + ", beta double, archivo string")
    ajustes_gem = spark.table("bronze_fragilidad_gem").select("archivo", "contenido").groupBy("archivo").applyInPandas(ajustar_gem, ESQ_GEM)
    frag = (base_frag.join(ajustes_gem, "archivo", "left")
            .select("tipologia", "taxonomia_gem", "pisos_gem", F.coalesce("im_gem", "im").alias("im"),
                    *[F.coalesce(f"{c}_gem", c).alias(c) for c in COLS_THETA], F.coalesce("beta_gem", "beta").alias("beta"), F.lit(FUENTE_FRAGILIDAD).alias("fuente")))
    escribir(frag, "silver_fragilidad_parametros")

## 10. Verificación

In [ ]:
# Borra la malla temporal; muestra cuarentena, inventario y calidad.
spark.sql("DROP TABLE IF EXISTS silver_tmp_vs30")
spark.table("silver_ondas_registros_cuarentena").groupBy("evento", "motivo_rechazo").count().orderBy("evento").show(truncate=False)
display(inventario("silver_"))
display(spark.table("silver_calidad_log").groupBy("tabla", "accion").agg(F.sum("n_filas").alias("filas")).orderBy("tabla"))